# Test domain samples — 97 URL / 97 hostname

Chạy các ô từ trên xuống (**Run All**). Kaggle cần bật **Internet**; chỉ dùng CPU.
Notebook chọn một URL gốc cho mỗi hostname,
lấy từ `links_corpus.parquet` (4.394.718 dòng; seed 42).
Có 97 mẫu cũ và 0 mẫu bổ sung.
Không cần clone repo, tải Parquet hay upload code Python.

Dùng snapshot crawler và probe hiện tại: tải trực tiếp URL mẫu, retry,
giới hạn tốc độ, tải HTML/PDF, trích xuất và kiểm tra chất lượng (QC).
Kết quả sẽ khác theo thời điểm và môi trường mạng.
HTTP User-Agent là Mozilla/Chrome; Accept và Accept-Language được đặt theo trình duyệt.
User-Agent không đảm bảo hết 403; kết quả thực tế được giữ nguyên.
Bảng ưu tiên domain lỗi xếp theo số URL hostname trong toàn dataset giảm dần,
không xếp theo số lỗi mẫu. Giữ riêng subdomain, số dòng nguồn chưa loại URL trùng.

- **VAO_DUOC**: URL cuối trả HTTP 2xx, kể cả khi trích xuất/QC thất bại.
- **KHONG_VAO_DUOC**: đã chạy nhưng không lấy được HTTP 2xx của trang đích.
- **CHUA_CHAY**: chưa có checkpoint, không tính là lỗi.

`robots_http_status` là HTTP của robots.txt; `http_status` là HTTP của trang.
Dừng do robots là kết quả của chính sách crawler, không chứng minh website
không mở được bằng trình duyệt. Nhóm ưu tiên có thêm đối chiếu Selenium.
Vài mẫu mỗi hostname chưa đại diện cho toàn bộ URL trên website.


Toàn bộ **97 hostname**, mỗi hostname một URL; giữ bộ mẫu reservoir seed 42 ban đầu.

            Chế độ này tải trực tiếp URL mẫu công khai bằng Requests với header trình duyệt;
            robots không chặn bước kiểm tra truy cập. Kết quả Requests và Selenium được giữ riêng.
            `BROWSER_TEST_MODE` mặc định thử Selenium cho URL Requests lỗi;
            `first_each_domain_and_failures` dùng khi muốn đối chiếu thêm mẫu đầu mỗi hostname.
            HTTP 2xx vẫn cần QC: trang lỗi trả 200 hoặc nội dung quảng cáo không phải bài sạch.


## 1. Cấu hình

Mặc định chạy toàn bộ URL đã nhúng. Có thể đặt `MAX_NEW = 10` để thử trước, rồi đổi
thành `None` và chỉ chạy lại ô crawl để làm tiếp. Checkpoint đã có, kể cả
lỗi, được bỏ qua. Muốn thử lại tất cả URL từ đầu, chạy lại ô cấu hình để
tạo thư mục mới. `RESUME_RUN_DIR` dùng để tiếp tục thư mục output đã có.


In [ ]:
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

WORKERS = 8
DELAY = 1.0
ATTEMPTS = 3
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
NETWORK_CHECK_DOMAINS = ('www.120ask.com', 'bingli.iiyi.com', 'article.iiyi.com')
BROWSER_TEST_MODE = 'failed_only'  # off / first_each_domain_and_failures / all
BROWSER_USER_AGENT = None  # None: dùng User-Agent thực của Chrome đang chạy
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None       # None: Selenium Manager tìm hoặc tải Chrome phù hợp
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
PAGE_ATTEMPTS = 3
MAX_NEW = None           # None: toàn bộ; 10: thêm tối đa 10 URL
RESUME_RUN_DIR = None    # Hoặc đường dẫn thư mục run đã có
RECOVERY_RUN_DIR = None  # Run cũ: tạo run mới, reuse raw 2xx và thử lại các URL lỗi

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python 3.11 trở lên.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
if type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('ATTEMPTS phải là số nguyên dương.')
if isinstance(DELAY, bool) or not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_DIR and RESUME_RUN_DIR:
    raise ValueError('Recovery tạo run mới; không đặt đồng thời RESUME_RUN_DIR.')
OUTPUT_BASE = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
               else Path('/content') if Path('/content').is_dir()
               else Path.cwd() / 'runs')
RUN_DIR = (Path(RESUME_RUN_DIR).expanduser().resolve() if RESUME_RUN_DIR else
           OUTPUT_BASE / ('domain_probe_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                          + '_' + uuid.uuid4().hex[:6]))
if RESUME_RUN_DIR and not (RUN_DIR / 'manifest.json').is_file():
    raise ValueError('RESUME_RUN_DIR phải chứa manifest.json của notebook này.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)

DIRECT_PUBLIC_PROBE = True
NETWORK_CHECK_DOMAINS = ('www.cnkang.com', 'www.120ask.com', 'www.familydoctor.com.cn', 'ask.39.net', 'zysjonline.com', 'www.a-hospital.com', 'suckhoecongdongonline.vn', 'zhongyibaodian.net', 'suckhoedoisong.vn', 'nhathuoclongchau.com.vn', 'www.zydcd.com', 'thanhnien.vn', 'www.wujue.com', 'www.youlai.cn', 'laodong.vn', 'baby.39.net', 'phunusuckhoe.giadinhonline.vn', 'www.vinmec.com', 'bingli.iiyi.com', 'medlatec.vn', 'qihuangzhishu.com', 'jbk.39.net', 'giadinhonline.vn', 'baohaiphong.vn', 'baonghean.vn', 'vietnamnet.vn', 'care.39.net', 'test.pmphai.com', 'baodanang.vn', 'tiemchunglongchau.com.vn', 'fitness.39.net', 'baocantho.com.vn', 'hellobacsi.com', 'woman.39.net', 'food.39.net', 'baoangiang.com.vn', 'fk.39.net', 'vov.vn', 'cancer.39.net', 'khoahocphothong.vn', 'jb39.com', 'youmed.vn', 'vov2.vov.vn', 'nk.39.net', 'phuyen.baodaklak.vn', 'www.msdmanuals.cn', 'heart.39.net', 'article.iiyi.com', 'suckhoeviet.org.vn', 'baoquangtri.vn', 'pmc-ecm-healthblog.beta.pharmacity.io', 'man.39.net', 'nhandan.vn', 'baovinhlong.com.vn', 'gan.39.net', 'tamanhhospital.vn', 'tnb.39.net', 'wei.39.net', 'tuoitre.vn', 'baidianfeng.familydoctor.com.cn', 'www.qdnd.vn', 'www.vietnamplus.vn', 'baogialai.com.vn', 'shen.39.net', 'ek.39.net', 'tienphong.vn', 'thaythuocvietnam.vn', 'baoquangninh.vn', 'bachmai.gov.vn', 'www.baidu.com', 'baochinhphu.vn', 'dantri.com.vn', 'gk.39.net', 'vnexpress.net', 'baolangson.vn', 'www.sggp.org.vn', 'gc.39.net', 'pf.39.net', 'health.people.com.cn', 'baophutho.vn', 'hanoimoi.vn', 'benhviennhitrunguong.gov.vn', 'baotayninh.vn', 'sj.39.net', 'baothanhhoa.vn', 'byby.39.net', 'benhvienvietduc.org', 'www.pharmacity.vn', 'www.familydoctor.cn', 'qy.familydoctor.com.cn', 'v.familydoctor.com.cn', 'ask.familydoctor.com.cn', 'mega.vietnamplus.vn', 'yanglao.familydoctor.com.cn', 'fk.99.com.cn', 'special.vietnamplus.vn', 'ypk.familydoctor.com.cn')


## 2. Cài dependencies

Cài các package crawler cần trong môi trường hiện tại. Phiên bản thực tế
và log pip được lưu trong output. Khi resume, dùng phiên bản đã lưu.


In [ ]:
import subprocess
import importlib.metadata

packages = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'ipython']
if DIRECT_PUBLIC_PROBE and BROWSER_TEST_MODE != 'off':
    packages.append('selenium')
requirement_file = RUN_DIR / 'notebook_requirements.txt'
install_args = ['-r', str(requirement_file)] if requirement_file.exists() else packages
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding='utf-8', errors='replace',
)
(RUN_DIR / 'pip_install.log').write_text(installation.stdout or '', encoding='utf-8')
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('Cài dependencies thất bại; xem pip_install.log.')
versions = {name: importlib.metadata.version(name) for name in packages}
if not requirement_file.exists():
    requirement_file.write_text(''.join(f'{p}=={v}\n' for p, v in versions.items()), encoding='utf-8')
print(json.dumps(versions, indent=2))


## 3. Danh sách mẫu và snapshot crawler

97 mẫu giữ nguyên ID/URL nguồn. Ô code có thể mở để kiểm tra danh sách
và snapshot. Crawler được kiểm tra SHA-256 trước khi import.


In [ ]:
SAMPLES = [{'id': 1826763,
  'url': 'http://www.cnkang.com/myjk/201202/480566.html',
  'domain': 'www.cnkang.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2843944,
  'url': 'https://www.120ask.com/question/38159402.htm',
  'domain': 'www.120ask.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3891869,
  'url': 'https://www.familydoctor.com.cn/tangniaobing/a/201702/1706297.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2211756,
  'url': 'https://ask.39.net/question/20931974.html',
  'domain': 'ask.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4182981,
  'url': 'https://zysjonline.com/articles/misc/16204/',
  'domain': 'zysjonline.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2992495,
  'url': 'https://www.a-hospital.com/w/%E4%B8%8A%E8%85%94%E9%9D%99%E8%84%89%E7%BB%BC%E5%90%88%E5%BE%81',
  'domain': 'www.a-hospital.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 459741,
  'url': 'https://suckhoecongdongonline.vn/tu-vi-thang-5-2021-am-lich-cua-12-con-giap-chi-tiet-nhat',
  'domain': 'suckhoecongdongonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4065823,
  'url': 'https://zhongyibaodian.net/a/142698.html',
  'domain': 'zhongyibaodian.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 546923,
  'url': 'https://suckhoedoisong.vn/tai-sao-ngay-nao-cung-can-chong-nang-cho-da-16995354.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 268030,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/thong-tin-can-biet-ve-phuong-phap-phau-thuat-lay-duong-ro-luan-nhi-59283.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2065346,
  'url': 'http://www.zydcd.com/zhongyi/zy20700zh/',
  'domain': 'www.zydcd.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 651969,
  'url': 'https://thanhnien.vn/tong-hoi-y-hoc-vn-de-nghi-cho-bac-si-vu-tai-bien-chay-than-duoc-tai-ngoai-185676141.htm',
  'domain': 'thanhnien.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2040931,
  'url': 'http://www.wujue.com/zyzy/zycd/p/200704/14378.html',
  'domain': 'www.wujue.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4013480,
  'url': 'https://www.youlai.cn/yyk/article/608596.html',
  'domain': 'www.youlai.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 154292,
  'url': 'https://laodong.vn/suc-khoe/4-thuc-uong-giai-nhiet-giup-giam-trao-nguoc-axit-1205578.ldo',
  'domain': 'laodong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 772903,
  'url': 'http://baby.39.net/a/141222/4540983.html',
  'domain': 'baby.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 304343,
  'url': 'https://phunusuckhoe.giadinhonline.vn/nhung-loi-ich-bat-ngo-cua-nam-oi-voi-nguoi-benh-tim-mach-c12a511869.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 750899,
  'url': 'https://www.vinmec.com/vie/bai-viet/roi-loan-tieu-tien-o-phu-nu-mang-thai-vi',
  'domain': 'www.vinmec.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2501085,
  'url': 'https://bingli.iiyi.com/show/64998-1.html',
  'domain': 'bingli.iiyi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 176969,
  'url': 'https://medlatec.vn/tin-tuc/bi-mat-cam-xuc-khi-quan-he-co-nguyen-nhan-la-do-dau-s78-n22073',
  'domain': 'medlatec.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 895407,
  'url': 'http://qihuangzhishu.com/764/120.htm',
  'domain': 'qihuangzhishu.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2525727,
  'url': 'https://jbk.39.net/shoushu/xws/',
  'domain': 'jbk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 108128,
  'url': 'https://giadinhonline.vn/giup-me-bau-ngu-ngon-voi-10-meo-don-gian-d19025.html',
  'domain': 'giadinhonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 58905,
  'url': 'https://baohaiphong.vn/tat-ca-benh-nhan-covid-19-cua-hai-duong-deu-da-xuat-vien-353669.html',
  'domain': 'baohaiphong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 67234,
  'url': 'https://baonghean.vn/an-uong-lam-am-co-the-10014055.html',
  'domain': 'baonghean.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 686659,
  'url': 'https://vietnamnet.vn/cach-dap-mat-na-cham-soc-da-mat-phu-hop-voi-tung-loai-da-2018837.html',
  'domain': 'vietnamnet.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 801009,
  'url': 'http://care.39.net/a/170224/5230598.html',
  'domain': 'care.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2541313,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=0001AA100000000EO44K',
  'domain': 'test.pmphai.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 34798,
  'url': 'https://baodanang.vn/khong-chu-quan-voi-benh-glaucoma-3239403.html',
  'domain': 'baodanang.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 674464,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/vi-khuan-nhuom-soi-la-gi-y-nghia-trong-chan-doan-nhiem-trung',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 818281,
  'url': 'http://fitness.39.net/a/180124/6017569.html',
  'domain': 'fitness.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 23544,
  'url': 'https://baocantho.com.vn/tai-sao-chung-ta-e-oc-moi-khi-cao-mong-tay-len-bang-a72145.html',
  'domain': 'baocantho.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 125814,
  'url': 'https://hellobacsi.com/nuoi-day-con/nhi-khoa/van-de-nhi-khoa-khac/tre-co-ham-rang-bi-o-vang/',
  'domain': 'hellobacsi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 914722,
  'url': 'http://woman.39.net/a/140915/4472540.html',
  'domain': 'woman.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 844242,
  'url': 'http://food.39.net/a/2010821/1451769.html',
  'domain': 'food.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 13933,
  'url': 'https://baoangiang.com.vn/who-canh-bao-nguy-co-lay-lan-cao-cua-virus-chikungunya-a424815.html',
  'domain': 'baoangiang.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 836246,
  'url': 'http://fk.39.net/a/241010/i1trnyn.html',
  'domain': 'fk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 710247,
  'url': 'https://vov.vn/suc-khoe/nhung-thu-co-the-can-khi-ban-gia-di-post981896.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2507808,
  'url': 'https://cancer.39.net/a/210407/8822699.html',
  'domain': 'cancer.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 140282,
  'url': 'https://khoahocphothong.vn/phat-hien-khang-the-co-the-ngan-ngua-hau-het-cac-chung-hiv-240586.html',
  'domain': 'khoahocphothong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2516720,
  'url': 'https://jb39.com/jibing/ZiGongNeiMoZengSheng264317.htm',
  'domain': 'jb39.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 760817,
  'url': 'https://youmed.vn/tin-tuc/chi-nha-khoa-va-tam-nuoc-dung-cu-nao-tot-hon/',
  'domain': 'youmed.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 717975,
  'url': 'https://vov2.vov.vn/suc-khoe/phong-va-dieu-tri-benh-viem-bo-mi-mat-24968.vov2',
  'domain': 'vov2.vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 871353,
  'url': 'http://nk.39.net/a/150617/4640167.html',
  'domain': 'nk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 312382,
  'url': 'https://phuyen.baodaklak.vn/95/141195/mit-tinh-huong-ung-tuan-le-truyen-thong-ve-phong-chong-khang-thuoc/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3980742,
  'url': 'https://www.msdmanuals.cn/home/skin-disorders/nail-disorders',
  'domain': 'www.msdmanuals.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 860914,
  'url': 'http://heart.39.net/a/190801/7350145.html',
  'domain': 'heart.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2136074,
  'url': 'https://article.iiyi.com/detail/416303.html',
  'domain': 'article.iiyi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 565289,
  'url': 'https://suckhoeviet.org.vn/cong-dung-tuyet-voi-cua-hoa-dau-biec-voi-suc-khoe-va-nhung-luu-y-khi-su-dung-8483.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 88645,
  'url': 'https://baoquangtri.vn/suc-khoe/202207/50-benh-nhan-covid-19-dang-dieu-tri-tai-benh-vien-2202255/',
  'domain': 'baoquangtri.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 319666,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/hoat-chat-amitriptylin/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 868618,
  'url': 'http://man.39.net/a/220924/8244035.html',
  'domain': 'man.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 202493,
  'url': 'https://nhandan.vn/post-853313.html',
  'domain': 'nhandan.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 97029,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202511/dung-lieu-an-thuc-pham-moc-nguy-hiem-khon-luong-5310d8b/',
  'domain': 'baovinhlong.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 852556,
  'url': 'http://gan.39.net/a/260621/x3wnanf.html',
  'domain': 'gan.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 570069,
  'url': 'https://tamanhhospital.vn/tin-tuc/cuu-song-be-so-sinh-bi-thoat-vi-hoanh-the-tui/',
  'domain': 'tamanhhospital.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 905348,
  'url': 'http://tnb.39.net/a/130607/4186274.html',
  'domain': 'tnb.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 909349,
  'url': 'http://wei.39.net/a/110821/1781775.html',
  'domain': 'wei.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 681131,
  'url': 'https://tuoitre.vn/loan-thi-truong-affiliate-ai-chiu-trach-nhiem-chat-luong-san-pham-20260130000926371.htm',
  'domain': 'tuoitre.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2475161,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201511/856792.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 722587,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/covid-hom-nay-12-5-them-2-439-ca-mac-moi-1-benh-nhan-tu-vong-727956',
  'domain': 'www.qdnd.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 732544,
  'url': 'https://www.vietnamplus.vn/to-chuc-y-te-the-gioi-dua-ra-khuyen-nghi-moi-ve-dich-cum-gia-cam-post942110.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 44335,
  'url': 'https://baogialai.com.vn/gia-lai-huong-ung-ngay-asean-phong-chong-benh-sot-xuat-huyet-post280882.html',
  'domain': 'baogialai.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 901143,
  'url': 'http://shen.39.net/a/140114/4326874.html',
  'domain': 'shen.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 812548,
  'url': 'http://ek.39.net/a/20101018/1506617.html',
  'domain': 'ek.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 678742,
  'url': 'https://tienphong.vn/weilaiya-white-truffle-2-nam-lien-tiep-la-san-pham-dau-goi-tot-nhat-nam-tai-elle-beauty-awards-2025-post1735881.tpo',
  'domain': 'tienphong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 660759,
  'url': 'https://thaythuocvietnam.vn/hen-phe-quan-va-nhung-dieu-khong-the-bo-qua/',
  'domain': 'thaythuocvietnam.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 84125,
  'url': 'https://baoquangninh.vn/tap-huan-cong-tac-giam-sat-phong-chong-dich-benh-do-vi-rut-nipah-niv-3394596.html',
  'domain': 'baoquangninh.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 1143,
  'url': 'https://bachmai.gov.vn/bai-viet/canh-bao-nguoi-lon-cung-dinh-thuy-dau?id=1e8dc2c0-88db-e42e-d5db-cb9137df465f',
  'domain': 'bachmai.gov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3160976,
  'url': 'https://www.baidu.com/bh/dict/ydxx_8424738550787796805?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 26481,
  'url': 'https://baochinhphu.vn/cac-dia-phuong-can-day-nhanh-tien-do-tiem-chung-vaccine-covid-19-10222060115443633.htm',
  'domain': 'baochinhphu.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 99635,
  'url': 'https://dantri.com.vn/suc-khoe/dung-thuc-pham-chuc-nang-nhu-the-nao-de-khong-loi-bat-cap-hai-ky-4-20251009161639294.htm',
  'domain': 'dantri.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 854825,
  'url': 'http://gk.39.net/a/090624/900447.html',
  'domain': 'gk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 702221,
  'url': 'https://vnexpress.net/bo-y-te-virus-nipah-co-tiem-nang-gay-dai-dich-5041215.html',
  'domain': 'vnexpress.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 64079,
  'url': 'https://baolangson.vn/thanh-tra-kiem-tra-kiem-nghiem-san-pham-duoc-tiep-nhan-xu-ly-cac-thong-tin-ve-thuoc-gia-5044678.html',
  'domain': 'baolangson.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 724965,
  'url': 'https://www.sggp.org.vn/ho-tro-con-dao-thuc-hien-phau-thuat-noi-soi-co-ke-hoach-post817576.html',
  'domain': 'www.sggp.org.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 853240,
  'url': 'http://gc.39.net/a/141221/4540391.html',
  'domain': 'gc.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 876479,
  'url': 'http://pf.39.net/a/2011418/1666222.html',
  'domain': 'pf.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 857643,
  'url': 'http://health.people.com.cn/n1/2026/0701/c14739-40751470.html',
  'domain': 'health.people.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 80524,
  'url': 'https://baophutho.vn/4-mon-pho-bien-tren-mam-co-cung-ram-thang-gieng-co-y-nghia-mang-lai-may-man-248768.htm',
  'domain': 'baophutho.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 118141,
  'url': 'https://hanoimoi.vn/dung-de-thuc-pham-ngay-tet-thanh-ganh-nang-cho-gan-ruot-734057.html',
  'domain': 'hanoimoi.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 98260,
  'url': 'https://benhviennhitrunguong.gov.vn/thuoc-diet-chuot-va-nguy-co-xuat-huyet-o-tre-em.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 90572,
  'url': 'https://baotayninh.vn/hanh-trinh-tan-hien-cua-nhung-thay-thuoc-uu-tu-140074.html',
  'domain': 'baotayninh.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 904470,
  'url': 'http://sj.39.net/a/140814/4447769.html',
  'domain': 'sj.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 91319,
  'url': 'https://baothanhhoa.vn/bac-sy-benh-vien-108-da-thuc-hien-thanh-cong-300-ca-ghep-gan-267964.htm',
  'domain': 'baothanhhoa.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 794284,
  'url': 'http://byby.39.net/a/230524/13823332.html',
  'domain': 'byby.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 98535,
  'url': 'https://benhvienvietduc.org/chuyen-gia-benh-vien-huu-nghi-viet-duc-tu-van-benh-ly-di-dang-dong-tinh-mach-nao.html',
  'domain': 'benhvienvietduc.org',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 720463,
  'url': 'https://www.pharmacity.vn/tang-sinh-mon-sau-khi-sinh-lieu-co-nen-khau-ngay-lap-tuc.htm',
  'domain': 'www.pharmacity.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3530740,
  'url': 'https://www.familydoctor.cn/a/201309/512842.html',
  'domain': 'www.familydoctor.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 900894,
  'url': 'http://qy.familydoctor.com.cn/a/201405/609715.html',
  'domain': 'qy.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2551069,
  'url': 'https://v.familydoctor.com.cn/a/201509/816823.html',
  'domain': 'v.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 766444,
  'url': 'http://ask.familydoctor.com.cn/zt/mn.html',
  'domain': 'ask.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 199047,
  'url': 'https://mega.vietnamplus.vn/thai-huong-nu-anh-hung-dam-me-lam-cho-dat-no-hoa-5415.html',
  'domain': 'mega.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2131080,
  'url': 'http://yanglao.familydoctor.com.cn/zt/scsjtfz.html',
  'domain': 'yanglao.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 836542,
  'url': 'http://fk.99.com.cn/news/410997.html',
  'domain': 'fk.99.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 583,
  'url': 'http://special.vietnamplus.vn/benh_vien_nhan_ai',
  'domain': 'special.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4030514,
  'url': 'https://ypk.familydoctor.com.cn/267067/',
  'domain': 'ypk.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0}]

SOURCE_PROFILE = {'rows': 4394718,
 'id_min': 583,
 'id_max': 4420561,
 'duplicate_ids': 0,
 'hosts': 97,
 'host_counts': {'special.vietnamplus.vn': 1,
                 'bachmai.gov.vn': 2783,
                 'baoangiang.com.vn': 10738,
                 'baocantho.com.vn': 12016,
                 'baochinhphu.vn': 2302,
                 'baodanang.vn': 14500,
                 'baogialai.com.vn': 3290,
                 'baohaiphong.vn': 16286,
                 'baolangson.vn': 1981,
                 'baonghean.vn': 15941,
                 'baophutho.vn': 1288,
                 'baoquangninh.vn': 2912,
                 'baoquangtri.vn': 5483,
                 'baotayninh.vn': 1198,
                 'baothanhhoa.vn': 782,
                 'baovinhlong.com.vn': 5113,
                 'benhviennhitrunguong.gov.vn': 1200,
                 'benhvienvietduc.org': 425,
                 'dantri.com.vn': 2063,
                 'giadinhonline.vn': 16729,
                 'hanoimoi.vn': 1283,
                 'hellobacsi.com': 11747,
                 'khoahocphothong.vn': 8143,
                 'laodong.vn': 31282,
                 'medlatec.vn': 24762,
                 'mega.vietnamplus.vn': 2,
                 'nhandan.vn': 5225,
                 'nhathuoclongchau.com.vn': 79598,
                 'phunusuckhoe.giadinhonline.vn': 27577,
                 'phuyen.baodaklak.vn': 6252,
                 'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                 'suckhoecongdongonline.vn': 154503,
                 'suckhoedoisong.vn': 85823,
                 'suckhoeviet.org.vn': 5627,
                 'tamanhhospital.vn': 4261,
                 'thanhnien.vn': 72111,
                 'thaythuocvietnam.vn': 2927,
                 'tiemchunglongchau.com.vn': 13281,
                 'tienphong.vn': 3021,
                 'tuoitre.vn': 3893,
                 'vietnamnet.vn': 15543,
                 'vnexpress.net': 2055,
                 'vov.vn': 9784,
                 'vov2.vov.vn': 6899,
                 'www.pharmacity.vn': 69,
                 'www.qdnd.vn': 3700,
                 'www.sggp.org.vn': 1853,
                 'www.vietnamplus.vn': 3695,
                 'www.vinmec.com': 26129,
                 'youmed.vn': 7189,
                 'ask.familydoctor.com.cn': 3,
                 'baby.39.net': 27621,
                 'byby.39.net': 439,
                 'cancer.39.net': 8415,
                 'care.39.net': 15134,
                 'ek.39.net': 3222,
                 'fitness.39.net': 12066,
                 'fk.39.net': 9812,
                 'fk.99.com.cn': 1,
                 'food.39.net': 11024,
                 'gan.39.net': 5027,
                 'gc.39.net': 1746,
                 'gk.39.net': 2059,
                 'health.people.com.cn': 1348,
                 'heart.39.net': 5965,
                 'man.39.net': 5320,
                 'nk.39.net': 6650,
                 'pf.39.net': 1538,
                 'qihuangzhishu.com': 23637,
                 'qy.familydoctor.com.cn': 40,
                 'shen.39.net': 3223,
                 'sj.39.net': 942,
                 'tnb.39.net': 4178,
                 'wei.39.net': 3894,
                 'woman.39.net': 11687,
                 'www.120ask.com': 918479,
                 'www.cnkang.com': 963438,
                 'www.wujue.com': 53341,
                 'www.zydcd.com': 78680,
                 'yanglao.familydoctor.com.cn': 2,
                 'article.iiyi.com': 5641,
                 'ask.39.net': 337375,
                 'baidianfeng.familydoctor.com.cn': 3798,
                 'bingli.iiyi.com': 24990,
                 'jb39.com': 7306,
                 'jbk.39.net': 19142,
                 'test.pmphai.com': 15120,
                 'v.familydoctor.com.cn': 10,
                 'www.a-hospital.com': 169339,
                 'www.baidu.com': 2631,
                 'www.familydoctor.cn': 44,
                 'www.familydoctor.com.cn': 447453,
                 'www.msdmanuals.cn': 6173,
                 'www.youlai.cn': 46119,
                 'ypk.familydoctor.com.cn': 1,
                 'zhongyibaodian.net': 146700,
                 'zysjonline.com': 243347},
 'first_1000_host_counts': {'special.vietnamplus.vn': 1, 'bachmai.gov.vn': 999},
 'sample_host_counts': {'article.iiyi.com': 1,
                        'ask.39.net': 1,
                        'ask.familydoctor.com.cn': 1,
                        'baby.39.net': 1,
                        'bachmai.gov.vn': 1,
                        'baidianfeng.familydoctor.com.cn': 1,
                        'baoangiang.com.vn': 1,
                        'baocantho.com.vn': 1,
                        'baochinhphu.vn': 1,
                        'baodanang.vn': 1,
                        'baogialai.com.vn': 1,
                        'baohaiphong.vn': 1,
                        'baolangson.vn': 1,
                        'baonghean.vn': 1,
                        'baophutho.vn': 1,
                        'baoquangninh.vn': 1,
                        'baoquangtri.vn': 1,
                        'baotayninh.vn': 1,
                        'baothanhhoa.vn': 1,
                        'baovinhlong.com.vn': 1,
                        'benhviennhitrunguong.gov.vn': 1,
                        'benhvienvietduc.org': 1,
                        'bingli.iiyi.com': 1,
                        'byby.39.net': 1,
                        'cancer.39.net': 1,
                        'care.39.net': 1,
                        'dantri.com.vn': 1,
                        'ek.39.net': 1,
                        'fitness.39.net': 1,
                        'fk.39.net': 1,
                        'fk.99.com.cn': 1,
                        'food.39.net': 1,
                        'gan.39.net': 1,
                        'gc.39.net': 1,
                        'giadinhonline.vn': 1,
                        'gk.39.net': 1,
                        'hanoimoi.vn': 1,
                        'health.people.com.cn': 1,
                        'heart.39.net': 1,
                        'hellobacsi.com': 1,
                        'jb39.com': 1,
                        'jbk.39.net': 1,
                        'khoahocphothong.vn': 1,
                        'laodong.vn': 1,
                        'man.39.net': 1,
                        'medlatec.vn': 1,
                        'mega.vietnamplus.vn': 1,
                        'nhandan.vn': 1,
                        'nhathuoclongchau.com.vn': 1,
                        'nk.39.net': 1,
                        'pf.39.net': 1,
                        'phunusuckhoe.giadinhonline.vn': 1,
                        'phuyen.baodaklak.vn': 1,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 1,
                        'qihuangzhishu.com': 1,
                        'qy.familydoctor.com.cn': 1,
                        'shen.39.net': 1,
                        'sj.39.net': 1,
                        'special.vietnamplus.vn': 1,
                        'suckhoecongdongonline.vn': 1,
                        'suckhoedoisong.vn': 1,
                        'suckhoeviet.org.vn': 1,
                        'tamanhhospital.vn': 1,
                        'test.pmphai.com': 1,
                        'thanhnien.vn': 1,
                        'thaythuocvietnam.vn': 1,
                        'tiemchunglongchau.com.vn': 1,
                        'tienphong.vn': 1,
                        'tnb.39.net': 1,
                        'tuoitre.vn': 1,
                        'v.familydoctor.com.cn': 1,
                        'vietnamnet.vn': 1,
                        'vnexpress.net': 1,
                        'vov.vn': 1,
                        'vov2.vov.vn': 1,
                        'wei.39.net': 1,
                        'woman.39.net': 1,
                        'www.120ask.com': 1,
                        'www.a-hospital.com': 1,
                        'www.baidu.com': 1,
                        'www.cnkang.com': 1,
                        'www.familydoctor.cn': 1,
                        'www.familydoctor.com.cn': 1,
                        'www.msdmanuals.cn': 1,
                        'www.pharmacity.vn': 1,
                        'www.qdnd.vn': 1,
                        'www.sggp.org.vn': 1,
                        'www.vietnamplus.vn': 1,
                        'www.vinmec.com': 1,
                        'www.wujue.com': 1,
                        'www.youlai.cn': 1,
                        'www.zydcd.com': 1,
                        'yanglao.familydoctor.com.cn': 1,
                        'youmed.vn': 1,
                        'ypk.familydoctor.com.cn': 1,
                        'zhongyibaodian.net': 1,
                        'zysjonline.com': 1}}

BASELINE_DOMAINS = {'www.cnkang.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.120ask.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'ask.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'zysjonline.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.a-hospital.com': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'suckhoecongdongonline.vn': {'error_type': 'not_tested',
                              'requested_additional': 0,
                              'selected_additional': 0,
                              'shortfall': 0},
 'zhongyibaodian.net': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'suckhoedoisong.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'nhathuoclongchau.com.vn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'www.zydcd.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'thanhnien.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'www.wujue.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'www.youlai.cn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'laodong.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baby.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'phunusuckhoe.giadinhonline.vn': {'error_type': 'not_tested',
                                   'requested_additional': 0,
                                   'selected_additional': 0,
                                   'shortfall': 0},
 'www.vinmec.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'bingli.iiyi.com': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'medlatec.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'qihuangzhishu.com': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'jbk.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'giadinhonline.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'baohaiphong.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'baonghean.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'vietnamnet.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'care.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'test.pmphai.com': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'baodanang.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'tiemchunglongchau.com.vn': {'error_type': 'not_tested',
                              'requested_additional': 0,
                              'selected_additional': 0,
                              'shortfall': 0},
 'fitness.39.net': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'baocantho.com.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'hellobacsi.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'woman.39.net': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'food.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'baoangiang.com.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'fk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vov.vn': {'error_type': 'not_tested', 'requested_additional': 0, 'selected_additional': 0, 'shortfall': 0},
 'cancer.39.net': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'khoahocphothong.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'jb39.com': {'error_type': 'not_tested',
              'requested_additional': 0,
              'selected_additional': 0,
              'shortfall': 0},
 'youmed.vn': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vov2.vov.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'nk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'phuyen.baodaklak.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'www.msdmanuals.cn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'heart.39.net': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'article.iiyi.com': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'suckhoeviet.org.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'baoquangtri.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'pmc-ecm-healthblog.beta.pharmacity.io': {'error_type': 'not_tested',
                                           'requested_additional': 0,
                                           'selected_additional': 0,
                                           'shortfall': 0},
 'man.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'nhandan.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baovinhlong.com.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'gan.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'tamanhhospital.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'tnb.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'wei.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'tuoitre.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baidianfeng.familydoctor.com.cn': {'error_type': 'not_tested',
                                     'requested_additional': 0,
                                     'selected_additional': 0,
                                     'shortfall': 0},
 'www.qdnd.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'www.vietnamplus.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'baogialai.com.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'shen.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'ek.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'tienphong.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'thaythuocvietnam.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'baoquangninh.vn': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'bachmai.gov.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.baidu.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'baochinhphu.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'dantri.com.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'gk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vnexpress.net': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'baolangson.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'www.sggp.org.vn': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'gc.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'pf.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'health.people.com.cn': {'error_type': 'not_tested',
                          'requested_additional': 0,
                          'selected_additional': 0,
                          'shortfall': 0},
 'baophutho.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'hanoimoi.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'benhviennhitrunguong.gov.vn': {'error_type': 'not_tested',
                                 'requested_additional': 0,
                                 'selected_additional': 0,
                                 'shortfall': 0},
 'baotayninh.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'sj.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'baothanhhoa.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'byby.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'benhvienvietduc.org': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'www.pharmacity.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'www.familydoctor.cn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'qy.familydoctor.com.cn': {'error_type': 'not_tested',
                            'requested_additional': 0,
                            'selected_additional': 0,
                            'shortfall': 0},
 'v.familydoctor.com.cn': {'error_type': 'not_tested',
                           'requested_additional': 0,
                           'selected_additional': 0,
                           'shortfall': 0},
 'ask.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'mega.vietnamplus.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'yanglao.familydoctor.com.cn': {'error_type': 'not_tested',
                                 'requested_additional': 0,
                                 'selected_additional': 0,
                                 'shortfall': 0},
 'fk.99.com.cn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'special.vietnamplus.vn': {'error_type': 'not_tested',
                            'requested_additional': 0,
                            'selected_additional': 0,
                            'shortfall': 0},
 'ypk.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0}}

EXPECTED_URL_COUNT = 97
EXPECTED_DOMAIN_COUNT = 97
BASELINE_ARCHIVE = ''
BASELINE_ARCHIVE_SHA256 = ''
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
CRAWLER_SHA256 = '8390facb2349996bbf79f37e890c1cd87fcd96ee010a3a8908acb9c108212206'
CRAWLER_PAYLOAD = 'c-qZ<ZIc|wapreM@IN%cVDt<)J3G4oXmY`!M1qurAPMq7P&&tAHSV41*==HGdf45w*yUnB$O`!+op>RgWrq}DhehceR!Ej8`*gY!;X8c*gZ~2&0!jUgdonAlx~jW+b{3-UOk0bd?y9V}tgL*i%yZ|?eK~m{NngEsJs4)u?a5j?n;Zr6WRe!$;945Iy@<1;U=r=dle`m5(-Hq$WKl9pW_v-J1<7m_AL6-t?%cUY9~o!qG#CuVi(-+*gF%o?=V?|1(QK9$QIVvx{Lx1qeMG;__U2KR$5L&WPA2h?3+Z}S(#5QZvraII$I)U^jFMqtO3#LiEQ@DFSG8s;-Pq6KX!LrTPF^~Uhl?Ub{U{#{)9HK?7xAbBP{fDDB-zy!<;^sjMSF23#YRyP7s)i%h4gRC0{*^}&UE+VX_QR5iz1nrp77|X!k|b8x}q8W-aE<M{Me83eRReAR{!8lcN!NF>PF?Kc{Cbfyt#SvR-Vqxk7-owOPA(&w8r;3e~}+qG#jPU@@Ltve4hqU{!E9r;-dUX>zEhGuxvg*DxV9UeBeX*#vb!xmJESJYCPP>MK+;P@>J=L;@w+mGV9PoKA$9oTQ*B~(}F8xy2f>S^@U^-zs|2MB%2>aS(e@g24Fg>Z@YvPiz4Q)!`fH0feldb*SD>y!ppp{#+*gtWD)@h^`MH$NcU_nE(YVth<fzpm#)8Y^|fn3KL`&lg^xb+=JWLUYWhwxnMCWCy1k(F6>O*Ub{<^25p49jy~_b!JazGMaQM{4cJTZh2>g|J_vNHmzjWcxx)+`bS}%X;#;dP%f=O~K4!#%<Z>8<v%09XfuWwuc0Q$e+jc6QYiBv(m*I#?#wHt%yzxWbbDLW8S7ax5DJ1tlg!&ZCi(MJOOm&F*@%%FXDmfmh@%m=m)+ubBj$7u$%Yqd)dqht?r)S`v1KoU&v=KIm+rKePh?tXkIH7ppI)ObL}TZG6O%<ZJuXH-^iyXichwZd#SY|}c8rIe>Z1}CkIJ3**A5H>U(73nk?4t9@<IOmo+fjt;{gUKuVQNAX$A+j{RHA=FU`jPi<WDC#?kjlImq__B|)G6ZW9H6)=4Q?=trZEHU(o^tcFb=!NW@B{;qmmJTw%RA%Vmc2kINjS>0&=5z&Hc5$W$}Cx4P)J3JyKPLXo!q6-F-_B(=!WHusd2z=XnbSF{D{ev_8s*Nz(s9G|4ecIuvF_f3w|<XSCm2;i4F?Jsq~KW+!PhDqF1}2{%F|@|LyKGy?-7(Jcmx$>@6kQ{9Pl7P~lKGL|sx5}|2XW6FFZYGV<?o3k(gq6c*5DZoocx1R!?K(8v}YH5nb-?~(a*BD?!VB#db9cMNc&S+yzl6T@34O;`FXJCv)OuJ>l@Y)xygdJ5z!0Lq$Z)O-ohlk?4)$bheS=`MRyR9tTzF7n}7rkE8+j){o1_HkA49yh3IXk}4IZ+@@b84OsyLpi%^H!TD@{lHy0Cm-ra2MKbU8bCcIl78%)B?j`Rx@1`djHYuE`CdYl5x<Yuj@gGqdXa7V|QtRF=XC;xuI>``zXP|^W|u=h+hKH&syPYivnjt5KRc(j);5&QJ^7R#yA&rQt{*t_tO+VqF@%^4%XJtYFAHUo-VRs>~vAR(v)IMd;ab2>*|c5v#aG4z%b_HZa&-xV=++Y5ur*RJk}4k!(>Dg3nU76s3TynKtQ_1(LByUkOv7k8SpTzDui(oPev`O-R8o}XX2&9IT*4SXPE*QBqOkLqkt9&+#gyerfo5u<!0E)C|BcBdo|iM!|Ld=eeV5ETm54Oi<;fpVln}v2Bwb&H6M2Q8AKZl$81=%P^R4lFE>8|3Sk~5ApP`i=_p=J@?4nJ(E^Az#8IY~=jw}P*SRg~y>Oi^c|T4v(CntKj|XpFe<g?(#eNDpdK4sikT1sLWSD>gGMdk`6h{wwa5*@@7NwIk7-OJE!ETK2_z@SweWL8r9_s8yctzU-3kd3*4fqA<Ryl{f_V&oB9xVxKap|rq+v;JdQ%u<;p%c!kAQMlxlVFCx+08%>0HQ-H+}3!7jXfoJ*xFy+FN%5oxvll}?XRvgF|bIS{a<Yr#^Kc&k6zC_W^F4tM)k+C6TJfZKQN{cFo?os6HOb|(b$Fd_U2AWB<(6{dA3!}o#4rSuu&UmzL>V`#q1tX|3D7f!LwkudOgoT?D8JyfU34OC~xo_w@`zPX)Md2p{2-bW+@_RC7?P6K_3rZdV3K~*67p#d77j{7O50Rl*P15vSfF`?+6*FT&}*3L7(Wz5)^xT0dWlTOY&5gIM!S7QI36@VYABs+MS>$?cUx`s4Ey7*K?-LjI4r)vV{15u2aW4er;y-+3Er^nOV6-{htf8T&4`$zb_(B_d5S0Bsq>p<`{$EaFAlXX<|xwvqk$x6ZJTrc?iU^4co3w%n;mLNAFa~fUPCFwc~E(4d-KZ<2}T-p#d$~G;;{i%n9hJ(PiCYlVqCEY3xjYZN-UL6F^!-uHjig`reJ^VChG#V>%zU%!Dz)+X|RGg{lqyN)IQMVL?mbLl4$f(0D*=2^sLeJseNg9JD=^#!|1j*ShE1TyO_1oTcxAT57}h1KEYGOdTGy0>p2cB`}+zWJY;bI{`Lve-cf1M^TV%1(_VP0=|f{g6MimvIazms3<bt1L#pQ8t`}E$~A~}0Oc!?&h___&*Exy=J9(vl3Uz_ezTO;y#ekb8RHmCL7~>xlF^z1M|)p>0#qwL98MN_auBymE@K>+mg$J1wydV|c%t~0CQbzBG4?1<(Uve$&-KHO0*S%vm(PZ3!rD8|wzJx)uG#VV^O?QEECpoxtUf7MO!x4VRkL=EYKy11i30i#-4~g6%YZY?$R1muPWVJ1nN+yEwKYDXjt%^iu6;7=L*rNtZ!73kD7_-#CzTrA#hh54mL{4Qq0G;YH6N-=ZNuBIEIlh}PxyTWhPnL-EuMlZr&uZG>e7r21I3@ITBvkf_CKTv!ZL!UYb_jD)agLY8|{iIFEbcbJUHB`e2xy~b9Cr_wi+=^DM$si)<a!By75fQWzJ)ME4&E4sRtX9VpfQg8MsAlE~cZWsZxd)OQ4-J&Y3GwxwAFIP6HkdqJn50>5ynIVAArK-h|Iz{o*29+<@x%a9yjNX`eywI!^Ye#4yd~i=5u^@kOs#*BsiTRLh2MV0UCkV~^+Q<RHeG*fHxW`kgIHk0b(?1C8ba%r~`#71O&V^wJnUSb{JnHcU;Qq03GeH9-Y9p<UcvCc&DR4h&u6Z|R(R4t4~;3~R!#6r~Vk5R|Tsz&l*uuCOL(8FssPfVrrg2rLB;vc*iF2^KAaxPFkNi`)@@O@+3tpu;<&(%U;OlSB$zLlr7iE}bx4*KMp*Uw}8*I(QzRJ4RhaLr-|b=e+U`fcTUFr8F^vz`2Mk^p)E|<E82faN4y7R5{fNqSvNJBns${N_5v>`PV9PRCt}$4_9AYC$V#y<Sd$X!=_%4wh-FU==Oj#UZf+%;kw!D1<&@K5+v9W&KA7=dc>n$@Crl#<xamIkZEJ5UTHUlBBjq3kM1ZYr90U#TBxE<XQh*C`?)XC%@Pbg{_F+?>315KvZm>yLC_9qo<PM<57BApr?P0M+V7ag*;WMA1SwI%o$~!SnSnH@^WHOgYR2V61pHIW5T$`yfy;Udk0;UIfJ8sk0;j=MJ;gwIu*w*Wq6n)jnW}PDUA7}zupJ4a%lwE)60LSez@9p6MHvPpn)ZpzYkYGEJ+y*ZIs_u|nbYSJ#{;>3h*<=K2mz_p^a^@=Mt#vziM_e4!oEg`;MLAcLwpxAIl{bS3uOirOQ3#eYCMPbt*~|0$4#e16C;5xO5>z9J;AFnj-q|NlJtMWNtEZo6*3B|t@G01FrJfcK@&{U%noqECB<L>A!;&a4jKf6%(VrKp?&C{U9KG3NG?GvGe1~g7jqI%y9T0yK_pWZb^7%MQklhBb5AfX>c%8G>Th(xQQ_=@2?V%%p%a`Zy8wiad|=+v5!Uh05GjY<KEwPD$nXWo{f$cmQ$8yBLcWmEZ3!Ug%Cn=v?qY;Z(0`_9K^bGPWOXQt2?MFk9FK^fcv$kRN1)GIT2C2xIp_+c%HUDeJkq&Q`6{BEVtpzf9VvxKVawM*TeDD!((8IV;VQ=b#JgH&x_}~Vm-O%RG*1e!Qd(2(&~v*z#cQ&JUV%m%!qc&i|70ou3;j4kyVm<*Wq>wbmC)?BX8={AoeipdyD}&XPfJMF%M+KAgLK;oSYsh#Z)GJ&SvJjmYH3R)&Gn%QW95-(rR$$tSbHL5X;idLG4h6hL1awCK}zyB)g>@Nz16hW#~8?oK{y4Dc`1!~jTk3u1(a(wAcSve8-O`W=6tc1fHYjb;`Nf1M)npF^S7m~`4u>Sw5ol9;`vrCt<S(7VXwG-u?whIwBb%V*8lv&gDN7)j<|5W&1m9ujo&CMp*{ns-2t#p&VVf5BbE3O>sV|IW0iP0;ws(I_Doe9f><Hy;1*+Zb#|OC<X}f|Msg4ACPnT5)5%g)IVKkIxK3Ez1ZAPF6k%Pg@;>RlAuY8}s|5dsXasGS=B<vyHzBjGJ--KD4qN!m=h21dhjh5Jyp082-!G<<4j3NRO8~_Bd^EoNcE8tsrgMJ%JU?9uPuewCi+baFZa&X`LqiCf`{}5hE^QENRqRO0b4sbQOMAI9ovMg3Z^m-G1WJ%01p-^D!ggsK<<hpnU0#8+$||qsU@>l|b;Mw@V6jADqO3K{?|Xa4E<n0(%wz;l<IKN!@k0Adz%d}yCVDrbElObR1|MhNprfEjwRw$5Q+z<qds;exKOenN*2aJrX&s#80huGg*v?^}FSLC~Q*R>9Ot;wwj3h;JJxgZev{jkVjx%CZZDEY{W$O3!$|!!~>0Z0ZBB{OHy2*SXk1guY_Qjp;-VV7P4fZDKZZv5m;PzxLW_dJ@HK4FlMF?jT%N`q~bh;>7!m4}L|7*5Vt>$^<2S}`+dNIOggFY$lSG6L%HynJozR_vY*S=!P1WROkkV9~q6_X>ynFnIjtJEg4M;x`L&4vUTN)7NXdg@+k?yfS2RIeKi-(CQ7YyuNcWObYZdICtv*(+xtSmUYbPE*uK;j02tc{<i(dFH@$cxnk4!u{)NK>~G>LGs*>a1w(++>UVg!nkKCqN!{4-T(=&G)k2)U(#Ls7-L-s%@(UbJpHessy&fLOGulK8+oq3Y&KRx+_TNc?vb8`A($n&63(iM4emrfiQ{>Tx>)^Mn=E5IqQkc`0H3U^c23Fk=qDlxPj{etYEbfc=CMhsL!P$me(cOHchX;jwPCta1JCyvt$<ID)G#qn`^PP^8wmeec5yw}RMtJiI9ICsv>b+~(JV-tv6*R!-k}vR?B@|jQ&H%UOTlfUP0F?F8_Y&}3L55ZxWAYYohe{Jn6Nq*`^R5&%wAI{E=ElTt1RZCQfobR>C%O#+Dn#5HS?r$|6yS5=E5YHa{1?$ns1wzKgFR*#xODoj^Qi|hn-?Sn!%Jt;7)>4pUL|Vip=no?o@V>#$ELA;w9cEjYOqZ)gmzMGmor}*h@OD)>v!7md*Mys<><bf4WT>2*Yj=u4$lJe<d7qM=P!kqN)M|a=Yx5tT=YjV#s$8x>sQAx{8uBB^ADyuv%7X3s~~%(4MDiS{u5fX}BC#w(^JnA!O+_uY%~>i1oa<`AjFc)Vr_@NffcM2CL-;HV|AVQF871F?K3Vb#EE2_)n1~Bhx=cd>1)=NeeM6BP|jAl=e>V(b8UdFLFpPyY!5yp3Yl_W#G@t`frV)RSu#P+<*>E9$y39!ED^-pY0WTPf|+*T3am6HnBH5p2Hk2&U(O2N0^fFe0#*L!A6jYmy_`I#Nf7%3+wngx@0$4lkV>Zv#uHR<=ClJ^#SxRN)w^A$XT+N%p5Nk`b;7j8B_QxxrVQ+THWHX2raIjs5(or&5V<wlzmhD$wrNjH8c|9U>sl{K6RrZJXZZV**Z2jIG91Tb`oko^>%xEYvU=)*aO1@9QD~v`8ZV7p#xA4#%sw88E?4Vs)=8BhHTIB*q}}fHOWL#ViH2pvMMhG@qOSCle-RnGYewrg61hy5?Jt<HU)aEWG=0n1Sy>n%j@VR6fBfwSgw34%`-2;zF-?2zjpe`V8bJ)p7Au(v#k{|JcxdHU1`X;t95=-#yYqK-?DyaW5Zx6(`k>K*gKC=7N_E^ir<{{aJ#ZFJEe`FrYmcjUuVSgblwWffw+!jmKH;1fwuzO?&&^REcgf(PIQT|e0RtKO`aEYs=Im)a4NV5)vdS(tE+-qGfLRJir}NXXy+w<H1ehB1A{-7O(PAKC{Au2>WNWY^%XuOb9NJ(b3whQPMX@#t7Ep&j%LWX{e}Z)H{N5n(9^}Y(~rs~4oU4ATW?}!V57HrF*wivbWT6TPW?~IG9DefrF}76adV3@1Y4(i3L~faw<UZDErZ_QmQO`r&ztQx(03TDE9%P{u>izjh3!>m%kaViuYY`ND>zW*7Z_g%dv!}<Z6V@yOM3+4`$I2FOFKfW71C|(l^$wJz7}bok@hQ|+9#_HjBra_Jo>g5NPC&2a9f)~<y2V5E9imO^+BnqNqHoa*HCz@53Z9zbS(SyWRM+WXP$KT-U*u<bZ(VtLoNAOkhcRnycl;E<v}tfWbOCbr(u!pp26Hf#{M$oKnRo5VIzAfY^5>Rll+c%TPq^1I+Un2R06+nsn^@-2v9&9u>HNywq7a#dp)U72ZQ!0(p!T%ppjFo%mo;{Ir&56khW*=iP&LMz5wO(h|^5)$nSkQ9o0jwY*FLF(>El{*|pN6iU-hYC~af3z_DJjBatk6zlH=W33?ZLU|}xwHtEj=`txV>=V|5l<<Of~^hRAKTa@Rf+{rG<x&YfQ0qrbPA9y|LmarmA-1r+b0xl0o>Hy7H31gtHDG(~hBR|uGxfI_xnbIaNIP01utsgtJl@~#o^zr4&uzAtKXUWPI>N}9&%;C3du4(Yq&1WBj+ZW?kL#d_vMl%Z*+eN#^*p;~XGwEDvGfbDy1G2*d7lKs#j<6WiZn`Ex3_Q9~xv}3eeA&{9Do#MFyfjK@eBMk8%_L9_9zH$zFs-65>0N}CZll-U`cn>L+qYxtUBQmN*sQu~T<EcK>O5AkPQo*8BmhlB=_!2omu}p6Juu+fXWOrqpx_J+HUTK77~d~dw%${{-v8SnmzaHqF)MA2Jw7Frdc>gyk;pF%on3xv66#iF2((m<ui8=1b;QAkjx}*@@rQ2|zy0n!45ek5ZqeGB{z3Kih63MS0hwAAXCMY{U>m*#{t|DzqM)vDQqy7CY6M#CaC=qunP|H0Xsta1e1hHf+Q!c5WUYe#b+)X5zARf)b{!}&fK2vvZF>_js_%yS=JKl}mLFOTijU?kBOna#TynEscZ0IWmRhLa_iBT-R@!zIo4(<jmf7{CP-WS7BgAzXQ@nu9;QaZZrMtHVr4%$%a;+87ASetAnY6o`m)eacZ-Ms58onRSzHMVVEhyBkSf*GO`HJYD*Y$`Mua;Y;(X6g43(u(9a`%|-35~}iUC`^gB++a{ugVq%ulnW*(v&OA^cw{kQ3+EAmIxTb2r93^eo~0Vw&s>)w5HNI(J4AEFU+^7f~Cug<C4|vjU|e3I)9Jn;_7pPAOZL>iwkH3?2}l#Y%J49mBwt>pT+ertZV1Bet#hN?wL|)zM>wa861@=eb-ghZ@GNd<GIM&NUP74>*;LV_7WV`WRr<RLEToMc1q`-(m)}D_g$g(E_TrOLG~Wis=V?bbG9oh<(ux>ROp?ag$8mrH=b7w!WFx*8aq$jb@h*;U(LR!)|7kfPL9j0pStKSs!fRsK9vP^|H9cAXb-XxQ>dr8fgQB2rC7QtIg;uAORGsM2CJ4t%_&i-U5yU#(3oZ!o;b!r38ExsYKL^n4PBv^<~_}2vlpzQY*vH%(Q$~2Jx1i3oFLqdq4BIo(JXwdSk{`$uSk5uXf|LaW6KJ^=8d1yERiKSyXde~=2Bo~0PUN`sPcec+RgFK>)BWGdCN*At-YtWPAz9hT!&}_J?J`BmPmJp_OF;|xwtGKt4FQc^%F}fl}lBgRB93}uS~k0=Xk$3N-OL2zKV`fy=*!t%{xz5*TQcM(L>U%?IpVF6V%Vhi6mAo?Gg^jiqhaXi@WGdH_A?Yhd|AL+TSCM{SHQpA-GNaFP-SN2d_HT)f;`bH|fV{UA8r?+41hYB`V^2b@f<rHCSfFU1q*LyU}i`eNOjwWy4&vneCxWY+_*3b~c5g?IpI#ZATK_@!ZI(`l+lSurIeYXjQ&PF1t{Z%Pj1yiepRUux1(xWQ9xrt$`-ju1pAbOOD}sIBdY>lHq3=OuEqMx7DQVuB!Hw_KRGzdh}<LowVrBMJ0#pp+CFQ$R7`0Nx@kwJDc(50sji0MJ;L39f1fDePNU1E3e9SzL<4a;_ChgDb}8XGjMM3ihSyf{<`F+#kzioSVq8P_dw&qW+Y1vb*zM0ktW-JS`6@rrc67Q8I3&_snScAkG0Rvu>8jxlRZXZ9AtZJ9FFp04N;_%u~IFU*5dX_T}=Ay&06Lj{6ku^c&^L%R75kJW1~p%Q#C^NZbe(SWFWqCx@1*Ut<^(OFq`DM-ub*_#m|JdLukk6O{Wq`b%IlUW<rZ1iucRFaea-5ZCo_RyL`!Hml$uv5{^|^Ig%#e2@M@ZGMvO~h}SlaKuz}JMaChbRZMWfD%_v5m@kIM*8)Snt~E*y7#|Y8yS2f8H~H@c{(F)CUgEz`(ceN`3WnO*fFV*4n>akN`Nae1ItyrtM-GqpcrZDfQuxX#jXuZ4yaT#N3?6I3T0kJ|<={1n;~3Ktj1p=lvMLZwCH7thui}{2n14&pN<HWTtO~B^y>n<?T8Z}LeALz2^#E?<frfIt<MJV5rWQOU#nFT}T{6xfUenVowLTy4P9#-fcv>&Y24uNF1h{9=)$L|stv0Q(-W^svSu-Cr5=_(5dZtv{<<b!$lY(l`7)%9%T4F9yry~5iOR)re_idcE%c2dlTrkn1I4aA>9^47ScJn;0GI6Si8kydsPP|<!;;;CUD>!x(V(3x#p*bDQ7rXi@C%Z4VaP1?uw)}On-;%4HrbmL+VXlHr2zl%v&Mk2k1R_@)6C7zN<`V@lBC=5Kc!5A$oEqyM&e<ntef|9Q=aW(YoIWDp%u<}&u>!l%r4C)0<GpOuLwaB1u=}nPm#%}$fx5A-X{~^zI9}ti)Ed)XU8lAs2LCXS`POF!8l(#>Oir}|Kur+`QO4;#0$D-$)N6b9esmOk_`MImKHIyQpWnj&S~o|>8=cLQb?e`YC+*hf9=kbeyK$DKy9{}S&N}i%HHteZ%bcCm8QM0v_=Hc4)}NTJKk?@X4a_)FiR^xw+r86+Dgw&l4Cuh}968tHDHzd!Ljq(|zOcb)MG*jTpxjP~Gs`6qspa{JINhuZqGu++2K^+(!~DqPZNRBUza8x48)WKzWs{Bh-tdSoUvbz!slh8fit{11YBZzpQJ!1gt5W^?+0NcovC=6NEmfP@8vdZJGgCxwCDXSZ#fjR8z)#vzYbt!oV1Q6C@ft;!+rvz*$*yC65^2jrqlLt4BESql$V*{Ytm@r4NeJ3!7-k2mO|rg!ahuu(`$RE}XXsGtTyU;aQn9ssPQl1O&%mK`#8>{yVDY?DWz1o8y4BcR^pLGFouJmXKaX4f<65KTEK@Lw7WY9Uwb@b2PzLU_O%!xYb8z__=7#6@#?@=VD_8G*`&#fH-hBRAcorv2LQgd_gr7X~6*ZmeXl0Ffu0|Px&k;>VH#8q~3%ela6M%#6v}~>hjgi)o_ZSMO&Dicbhmn*oH@+J+%N9-?ohN$A{!_}~n|R9BZurFOFMeT7aJ>?QK;fSS&EpAWl6XuZQjIG{C3&l|M#$|))klg^#llla?XW5X50wd=BQp#k-FA8U(ksr%z)MY$K$SSGZgA394$(Rq?|51G<B)NV;_!{EKzzLE)fd_H9Tl4a1+9~xBkU`j<F2)vWV=un1iB9dd5~Ekm{iSNzu*wP42Og^h_^=r{z)<nf-&=F6&g+rjz`AjPt1CT%B@XY8+>b}1Pbl{CIx8`wzoDNA2=zez2Jh=Dr_y2g%AM%n~wvv4jkW27d=g^?bkC>9WctnyKfSa?ga85G@nQUV6xyqBmkvVQV2zpqnn11Bk<28_~Hwj7dAE(3ILm~9A(I}0;x#C%_~@vXd#X=j<7Co!5Q@NzwMjnZiYLbYi-}Wy|(jYJFM;`)Vx}92W(bU^|u_g_d~R!W_RPM3r}C%G2V@w5rajGJ>Xy*ww)g6oE&ILpm+ChI?pL;wb2fYO@G*RsN`2j3#|b#uf4ZRvq!D%?cSn)4%g_xUSz996xw*YccCX#lrRMW!R8C?p{eCT8VHO=ncN?@oV2VCY%Pl(Py?-}N7rT<EQ3|cK0LG%^-Yzt<3+9_H~n5o(_4$V_nu#^!=>+x5vi^8LtRaF%T6-6?X#gI<-`^P#nSadiAcvOXEfbWH^Empi8!$7h$x4Whs(?_>OIF>){z3?ZL4G4IxY$-z@nAwMI;z?fbsAcOLR0KH2_u`XrfutLI889X4^+F%R=;}MlD?`X-zCsN)n=SbNkJJB4(Mxs;VRq_9uB$Yeo~4fyNnBve3s}bpq(TC~2(;2BJiA#cX%?(u3~7%ofj?8|H&vryB!)*hn4|D@-pr8-U91VIh(2>%nWV?al@%^Owt-B>-%hM4h4FUg#82ObiQ&#4)v47sNPV^oUSnVNF<8WDWq#`ZBiII%y&BW$9U4!(N4Wbk>%Xc&3KO3sMbeR|*|ou}IaV*fwr83F>b&c2cRT>7aoUA+%TmOUc)^2CwF%>e8fyTNTTDti$yIB&yL%n?>Uf;{2%!Jt^(--Q4PZB84(q!A%YID|wANf!TO{jm87&{lR9qlp0AeO}<0H2TPyUfVu#f@{xOJeO0|*3RZeUD6ArDrW7omP^DW%5~aKB`Y?58m98gK;-C84+6JldEzgFEs7wm$E&Y->@@g$@u@oVrrh8#~#|hX|b12apSjCAMs62Y`teH&6>yEdweaJAAs_Iyo*?eJVw_LL11UhDYh4b1Dl*>UcjHTLFqRaW#m@l9iUzb3}wK{fb6B0~nIP9U6m+-m+8jxkQEfwb!n>D^2R)s6Kc4<T<shLBmQ7Bel^k`ot@^0l;$_ci^{#r@DIwGJFH5lqk4V|EJ^s$Xc)EG1_OYKgCg!X`?RvNy=!K1+5hc_1&F7|pMIpGiK4_-Vo9*<Spv!T9&V$_g)!(}Sw3(~%@7(rlAX~cK#$N>bF8||kC39C@rI?j~SC5M*zek8dp$`(8|>7EA-BzIDX<!%Zwxea1+Cg`$!Gka5;n9*|tswZe0FgS9-n9{t=NPa4trVsih0puPq&Fg6y;SE-+2m2H{Euhq?mmx1Cxej~6_Jm|NW0k=Kt_ceg1^6T(3s}}QJBAj)8Z(TZ-AxOBOkJ$WJNpA_PZ(YH<}MlgaQnlr-}~{0-@Es-d%ynZy^ntN;a}eS%MZVM@7wpj|Iv>=dVfbN6dOPvBra%&cGv~M6oGWWLECltr*i|+VQXWv$1`SXSlw1L2z$Si7>Cf*Y`z>klM<&xSGZyHZmc#{dA<h1|CyRLqGDcqy^Tf?!e`}+oKj*~FiK|6WzCp$$=>Oiy|XWyr+3k8o_#NFQ-7L!qd7vFXl1~ukRu=<jxyGLBHlAOX2Je;i#Axg%T54nFd+#xpQcC<s=`8dROSeZo+dUtIE%#WlWST11UEH6T(k&3{>`@@{QNr)e*3!z-~0B1U;gCx|MnvyH4nb=o8RC4+mFBfH=n-$txvxDSD$?UU-<2J?tb$9e|YeBKmFu`-#z%oyPy90w-5g7Pd|PC$Mo&vAN-e(fA9~VzWe@zzy2Ei{^Wmu?~|YZ@WK24_USi&<~i=e-~aYk58nOT2S5DjgJ1rDVgI*xAAIvSpZx0&KK=EtAN<uf9{lpR^!CH=l3G2O?ce|47jr1t@8A7;uzT;HXNYyUz$>;9UfI9*3%tnhy@v(az5l^a(qM{j_Da)YszK`&;{_~V$&4^bxzM<FyD8L3883<loirVMHt1fw)Chy3ajBBzlb56}PBRIs&<VG`Dg+LlNtX|sZ50psUo?V(vS=_PhlxQt%qm#uY!5$E(G0k$f~T)WG!rKShfidSafNeKJc0pG;}vzHgN;kQQ)rRRr&oA^DtN`u)eM9#R4&96Za`jDU^N;h)>hfPnhe3Vr{m%(<tTwa6VfwIpoYorO!IV&&A$9psK;j?ySa_VlJT8q`Nea0o`nGToE!n7xoI4C%BF9kJcRFyIS?^2%c3sl%*h2N<CwJJS6iRk!kx)GxTkoB7$AgbSN8|+yhS0Ckv}sf`g@0}%ve_@;~w4l!+ZbZ4}bsT-~X?7AN=A6pcnG{<G=ga$KU!t55D=UI|cdl1-JI^{hA2f{kwmSU3ve5_mVsM)xCd4!e#9Ge>@7fY&-N!pVMVwqcnFQbHQ9FonNF=R4%U8?8EL!xvGxR1)Ki&7l`bCheM4J;lyj?fTPSQSdMtcE(C8y2azJ%3=HmAw;<o$`NPk?^XdCv`{YMoyEBaDaAl3$6=_})H3a-W>L61(938_lxt9v6UoxNi$W#oVT+`EBrhTO9G_fA&w9UBwmJ`!Rn#$t0RLTKIQaG;iw_C=avBGR`DcF?%2b<BTpl4NGiRK~{`><uWit>^*VU|$HE&H@5e-fK~ZfUw(ZNedxHYB34qDCt7>xjoZAClDsz$YaOuY9nf-^86@go03!lQP!Al}Ujmp$z2>qe7uv_;KSaRf8s(nGFdshz${q(DFbpt}0WUN~K#}<yBqgqVeRzphyIqG^OvPWN{1Fd9gzsc|+HPx_*c592iJ+$pFzcEErBW4HHRYpwcz37*3;-O2njz>Fai_C=2?*imRc%au52V(K+(?^*c#)s@}A|xB+#O1lO@0QoX?Jwm-X+G|N)atdxT0@!-1lL1dp&%n1;Q9Y)vpLG9V>K|Djiq*GLa>;%jXF+_MECrwh5_;!VaHhN69@GP@oC{fMJzV4Om4p6);%R8@fnZraualIdwT%`8Gh2=@7*c&*PMfz~Auiw}`L0VkAI0+@zL))em8Z;#!5anxM$i&K*3QcsB#w2gc6KYjRg9(>U?<(YP#f;rLNsgOZ%bUX69A2knIU{|OuDV@Gk+!ouE2T>6$n*&(6Ar?W$?YH)aZ+`qoCr5E0&G=sqj8b|$r#WslVHdjI#k^hgVih~ZcMv+S`n3erXfp6CI3uA0uYsX#-GxIAe?C5UO8CglpuqddTMdBEnY^QYlfap(v*M>G<&X0dT?U@3QqA5a$wz*A`X7}rR#58eeD{ZZA#AdB(oa~_<6ud1*uJ)C{J~uu4)afIj!_Aq@e$UsLX)isjV*B=rYcwt@G#gvFYS8@j;m%LByU7%Aw{*$l$7eX(NJizam+Yy)so9{nHB+B9fTpkLO!r(}AL?bRUow$Xag<^{tYTpyrM^BfN5@b?sc@9MC?I)t56}_L<^qBzp^Q$}(R)9i%#~9y7h=cBXH<D>Vu%-GofM;m0HCdSoA74k&qBgj3}<U0Dd!DM21$IDo`0jQqpkOEx3#E3<Dc5~nx`-3ha=Z?kgIGDULmIOS)<fs8SKUP2@3W!cIi`kKCG2OXuPC?ADovvjgn!b9qZW4ecU!HF5l>0dS3AzRvY^`=~hd2|rF2@T6S9)El)-H_H$zCCPK!z<-T4T)Hm+>o2|<~AhOOjb(zWJS<gmh*9GrzotP=og*tc<d{(z$RZ@dRrrzlYac-CeULansNkzmnv;Hr#!TT(7D{UuizWpk*Dn~yQJWgcEhjbDQ5?$8=Sr7b$GA6bQlj8*gY+(PIsf|wg<zLAIb*`C2f&(!Ql}JeK}WmzSy0T@3j_o=vHJ^+OaWQI$pVg$$7Io0-PhFAQ5TI7f~akuC(DO!LqcDBcOJlkKp$OLRUEt)KcT9iP>h{)EuNQ?YT+${Y0>NiL}?fKzEXUu^W^unJ<TVJw2Zp!GO#Iut*;0TP9n2>>=H+94#g$ho5f22z=TNi^fn(*(Y`loCCy#usW@K4g93?bEJ5ghUXIh@Xxcna9Ds#!da3jt_@U`qbsy5DHT4ouf3s?jKJUa0KqVGlIg)C@jE(|RfvRy;&O0YWw<~Av!)R{NFgr`hky(uTO(Ni3~3K|{H4RLEjTjLfYzyd%p3zEv{0Cj8Ft_cqbGNjs~!%Bkz*SmGbF#mWdb-~vq8Qpkx2nQ9pHCZi)k+3d7^D(lK05ad#lxRB}4n@X1fRcCNFGOZGqy2G0hK#)tnXnd!ec%&jfpBNXO{@T}sTqe1GNT!L=7&x$-jY5AW4uxv1HEPg7CB>!2H8DTNL`FQVO09m)aUDP%hr(B(2@QYN~RyFF0y2m9|#=>+!Ntn^S#nUV2uR;(78mkMHK2v^l?bmZMDuU&b0@X~+86%6b5wHN+9&qsZTM@c*xofgTw2@TJLif6;FH?R4z%OAJmltp^;nsrr$3Xx}KO>E>XZ|IVtVm6yp)+ynY#PClcfJ)AIUwL0vd99~aqNU4d9;);4@Uvqv|NlwHxAO{LYZ8c6GkMROI9g8e)yUG$n>e`4S&F$Q9d^mMmQ8R%Zi{PGU(S+@n-#dLWaF+ft6R%*T~`xjYfY%j%sQ#FRch>q%ZfcdcO}iT#hm$E$docyC+F(pP(CLtt0?Q5>Tsr=QED1->6WcRUu|x0kb-b|l5Rpc!)MDkZ2=)f%oOLTB`a$kDn^R7>$$U|^?jmPY=UL$Ast1e0ATm$oG%%%?uSF&&}|~?k`Y5RHC+ZKvwc-7lcr#ASEtsvFlA1PYK%5g_ln?X=9)A1NuQDqCDU}y4)QJh5H~Xxmf6ilwP@e^W^cL?nZkpOofU?JJ{d>gfo_unL8w-x1sZg9Tpf@V<%i=;in45Jio?`RQ#`<5uxqjqy-wMswx>8Y@*Fw1H4d0c`h~9!un0mOJSSpP{46;4n1ge^2Fz@ggOHV!(z{0k@~W#)U1d~9j)Zi+m?*V7CvoISmV;@VQ-dqB8+pFTvN}$3c{AdxCKTeUXG^3b8<c5AP8X-O?OGKXNO`wj8rU*}3Cq#YWf}8>Z-JZa=k(J^TOvtO#-(OYOl#Y!TQ1Gp+M0>wAWI)W1#uOkvodf?8f1c^rF^^JHY^ib;dY$t?JIIdy*k{sr*sFQWALWLN8lfRo2lI?xfG3bzyzlR$!uoX30E{7By)1HpO+gQUA2t9J07ahuIe(D<hZY`H=3sR+B)i#7mq(xrtWLRmbcBrLs>a(<Wmc1o8`oJ1ZF;WfB1mf7)FhkqwF}i!dIU)z45>i1ODkqB?V9KnK9z80#k~f4%K8y<K4azni3=*sH)F^18Q6$P4sKbzua-C%IXO5HR8A!)H|$zxqb*_LlED`whw*%Xn=2n@&r`JonCjL>z;ZVB{(wMpt21NzF_m&+oBFeZ~uV<4`2*kkRzQJLo2%Uw&6RjP^2uU3GMP)m!}7OqO<E2E*y=LXeQnve3K4pOn%oAYN0s{rW`9moj-F)O$$}p<^Y_-IGMD%&s+j|V~1V=PIO>nXVoSkD*=@TX8&o@a=FxV38O$}2U3KQ5hBSsW(a$xsp&ygc{*vby=t6Q@@rQvX$c)PexKK4G%#8W6LK_C-3nk2#>KyYQY`}0SWLrRWA?fa=1^e;V&}^*OBe@={lpzm-b`|j*j(&ophik_ra-t6$CF6DxaOY*Y;tTIDU$3|!t!OT<gO{qU8S)kx;>4vsa?7UxI~9%co^<RTUQJmpEyI^hY<jyODIsADeDa?L^EJGAB0;{6z7&0kCp8%;m{+J3*1P7r8kh}U!}?RN!y>zTQyi!z~Q6zE&wmgmmkQ5L(*F7GZuP=e9)sbh@1KgG`=%dK)4^8IT+gK&<hWs<KK;G*NWwCBoGxQ#hLsBa={ZPw1q##B&%{A8Bg`1K=^p@^8F8fSKuxM9DS9NNWy)6Ie8&TU%h&rZ%RNVb#5)VvVZ^XyEDkX7M7k+rm@S|K=*#cK8OEU$ahBRlgq*I-oMgp-oN|z_@J!|hV6g?<6LzXs3}e<02afpW7v)zfGx6nzkq-8;lgUNJXxr~{k#9d9?n0Uk!=*k=F0C`55OD>4PCf$|AT*;1&2U~3As|*JrNp5{g|PFBp5Bw$nO2S{~7v~dST%Sn#OwL{@wqHHtzl>x0@g+q%grfg`e>J@yr{j4d>m?R(JD>hIX#)-}~=Vnz-LeBoI9ja;jo92lqetIfU5c{@wT0Vg^IPi>ogNd-v~tZ&-&LGT5KYN|nOo-jB-QR#^W1kA4+VGT<ZjKmA>B3&6il-JrMk@BR}I1Rkp0;AQ2hI!B?OBp9_S#`YJ8JfUwuuLJf+?{TGj@6F3es%Y&Lo-szqJj0*;=L_vF3mn0Uft|q@y7UOV8NxddeI<}7%;tpJBKn@=-B6u`JHc1tU5)~HhgS1%0F{>RPVg!>bSL;yxjm?ejl1~$yq>!|wAt<iYinx(|J_>tPiTp=HiKbbNTHps$>#Xno#6Peq}(Tg`fW=%`dx`p`b{Z8m{*oEN53j%BfL3h(;K>XO}*IWyJn9+j%mQebHI#y?<GQJ!oT>7zmVhHG7cK7cbVp$4jTq%MSrucs)oMhQEP+-uYd@tJ%&6-AN}h7-Ji_%R%^F%1<v2D(IeH}GeZO~dFASjKObCw>GjvH-{?;HpfYz*)eWkeTCj=zAy4};vB|F59bs-mc;c{0xZ)iv`gUF3=Q`xq<a_u$wq=S}O|`n~N>kS^HENK2Y9gOz$IJS`3CEZms%GU{U`AO}Com&xnKb)(0woy2V8GXlLpdnpamA8s8BmnH*0G_$UV9~rIR-Sl;fSM=GHNMz63%>=QTQQ_S-`FUWmsbsR-S^6%4MUgFP27N^{P{|)W{mc3WVPJ3^*7me)x<7KBc`jMC-d3G8PFhV7!WqzS(OYlJ2kMqF9#w%<M1<V!p&2LlZK+h0(z0DdQLomahQEw}O9wL=Uuk)oHL{*|z}p7ezAZQrs*mP>zp4c0^>L=gI_X22*v2){}i~Rdw>(NGAsreZEs-pX1|d-VPi;!M?{!u#v)435)(+uA8-75`Eh1IAl!%M^mcR>I81MeWQ!#81AIg5+(gsX@oM;NXs%2=GJA`MqFCiN@$!qIXscNRxJ{gJqEGLwYz(hbhj0r7c8jXqINBB<mTrXF=V((t1s@`Y?)xokbQ64>o4yyw_#RuP!mD^uJ>~GDdsknCe(~pccZkfRWk6X%Mw+y9OdJ5jBqs}(Y`2QkM|TSoyQkV40O_EYPo6+Gr?`eSa`~qRPnCbiXK8i_L77mjo-5AMiT{KqOU;KXTiBDu}qb^b(T2kpWR)E`Zaz3y@=*=4R<7b_y3n&UGP|h|NX~OXd384#8&W(|Dj?noYp08?1hH7_si~)G$X{RcS<^e+nIhTOndITxW^%PoR35+y4|u{9PBZt&gRP9jb$E#ma`x+gd|v*15jPq<*=(#_xWs(IbunKNij0%hz4`mAF#1^U~D?OWFO>W-DrdxD~nzAE2mnBOhrMZBF=KCZ;9F@XwLSlU9PI?;eIkk5@qH><eWc7(xX<mwx;N2Xs%)ODQr`;&H)?;!np$i{NsyeAauo|p-=S&5!{Auc2Y#aWLA_7)1{u~S~15(uh%l#&w8DN!lXiOa_a5JlX*Y9O3}D-cx-|f&GMpTxx?kq(zyp1?mY=G118{bmMScGGsrB4xpEE6UZ_D`r#EshjDiO*_{#8jG2w$*UA<is;urunIonBOViF_ErVzMeAXAjXJ9swi3nnbV71jiPHj6Bd(*AXBUNLojMJmBPF0!k2o3X=OXz{gop?Ut9{ajvKsan<Qyr|xhYckv%kr}~i`Lssly3(4yk<>8AOG`#6kchE51q@wPIvorpeUO_^elX9h07h&lYjj8IAvyyW`!$G|*3>USVcQjQ2_7rr{cwm}Z&X2B-kdsuP9w6i(^uO;XYQoxgBGPSj&3Ap3I9XT^HwR$xp(uza>s6P=m%n#Dn<0}<N`fU0c#^EAhutl>PoD*c4=+hBqEwpG=yCk=5W=x04cpJsj_CYZT2F16<TBK(x`3ERt!kr6J-WyFyQbR1I+-D&u?CgNc5+Jo-R)TYwH4`7R%{GfAo?63$T`=KL'

PROBE_KIND = 'all_hosts_public_pages'
SAMPLING_METHOD = 'equal_host_reservoir'
PRIORITY_SHA256 = '7cb7f7900c8900e4bbcda855408ff94b5e978918cf0fd56b34079b9a217167d5'
PRIORITY_PAYLOAD = 'c-qZ9O^@70c6S5%4~%>uTkdZ4jI52^ay!tD#w%gHk|B+31hWHz-OcW%hfTKmF{ANd&N=6pTYw<99CFBMeaOK<{)hib-h1`xi!8EdBpZmZ@HEM)S5@z;-dEM$zkmPjHcC@(SI^UAG1^7z*ozj6xGcRzzTHJdQs&vzn-}?I85iCrj+SvzdP(NJiGQr)vMPsOzN?Zvi&F1xoW@yFZ^!rV-@o_Cs>rthBwN%)5ogtSRabQpL*MN#FDmb3Qvk4U^E^F%A1~@EFNR)JhB#^(SMgE<sN(lkn#^U#tWHyYd+!r|xBe;FsfQn6%IaYoRU7q?mkf0WZw-*&nvKeOzAJJ>fqr$Re^<#iW+-)$0+2Casf;Bkc=tn|WJCP1j8{dpRtrhfc!8KwD1h%#omR_a0V}!p$#=(Z-oAYO%A0xqu1NACsjfyalX*P4c;w&v<i+u`=g<N^U*G%0gMa=rAk?n%r=E|<c(|#y=`h;ufRGW6^YA^s`Rsl3?KXY-<7_hiV))?U1N!yEA6gJc-$dEE23A1OKW#=wuZBNuxKErly1d)13t)=T%d7-47V$_j26jHcW;eLFd!N8+y)3RS^Wr>=iy|+=^CVjaiz2#AV<3L~eleH|pv$<5k`z{2Rz-mC$7y~U7r{V$PF7x0CRtfU*&@d8Lyw`13m$w7Z)4)ia{Tt~H^(@hgpU7;7??5h{IX2NWc&{TeUxVz5cLFjBCjjg01MF~tVEL!8|Dp2O3xT5NbLcl>?$BhWo)kU!lUQF&!TM%qL`;O!WZiwdj1+3E|*1;t#WS#UE(Fa6~DYH_o&N7><?U%SY~AtlQ6Mu4x*ftj0FH_ObF)VFVegO;QSpX<6S1z2G)Ut?O^S((W4u^&@mLCjL{9C04%O<4N0{AAWHR6$tI9vScyF;EQ<|j<~Uo&A*$tob??xd=gTWOfK<q!6walPOuY*t_W2Ngz>>MmIH}@o84PYjPihSP5ZdYVU~>4m_rSwHXST5MMCI^fV3NRJtVY|UO4d;&c*Iu{>dioXshRs44fqcxHTS=)k}8eA`k^i>FY>ne7B&YW4MA{p*u@1bXPabek5mlkMeiamk`-&T-p)K)R8=sHV1*y+q+m{@UIUHt0yK79Zo(3@KQs<jakbc>a;jnX3xNumA}#E3&w;U{0z_jIv%Hv$yo0UHyWGHQZ-*RJT;K{`oSb~?F?djKfIzWZB1mX<k>oW}lLsEli7+u1d48V6zyq4@veTHq!Qb6gFc^Bctpw^TcVIZc-xbV8@Xgdi!LLBo%GN{gd>~jz=nKO36%u+87Zv`E>S}|3s{+Uv#@U50`3iQd_cTS=7Nadr73PHl+4JFMt$B)G?dc<sfsLjyNxVeOGENoN(tHu65_fXN0_B~k<wuFAalKmtae^k^oQ-&E9(oTRT>I~U)T3v}DzF1;zPV|DkmO<z{Mcy3Sdxt-(i)8jWvY!pU}L_BKD8tRmqg_F3N%HO)-h6BvW8R_h9)bSO&IFX=xNB%Wt3DQzy&p06<2zHSF#je06KucfWtJ&VlewaIJY|`StX1NbdqWom@{0Z%YeF_`e0Z2XG4;=ums`Cs`P4hQos~Y;AN*xURG8QUT>CP0%%ml<87YhRh}ga$pJWV__&SU2ZumOIW6xqt?{F5nQzB+mY_lkCPQ!hc%W7`&cF%_ql#1{E@J3S2HptA)P@TU@|ib2@V=TU6o^I!rg`Rs!4QxNTLWmUGc*m4K&>qE%k1GiTAb&r71~X4yevU?ZKEVZ9m5-wH?Nm#wUHUR{Oc$!WAWA+h$sgtHqVwN4OZkeSZMfq=sn^!5bnw}j(7CI-X)L@rZ%N{03$5&V(H+7GD2e?Jczn7RqW8?y>iG)0KppCx?L@(+@W;^R&rjIGg{ct6XS);q?q~Qcj+5y`|TzIY<lGD@Ki_+fV2@>2uZZgfPIoh2{x3_?f_#4dKnldo`dvgl|7HwN!H0xioP>$_!%!qA`RyzRii`(YzhTztH4Mz$#-?NK#eX9j@juMKEV25?eyeA4(z`$G-815aRngU=1crQNCV{!i(k-&xN0EOivgct+R;Fv@Da%v3CS<<o#Qesu6-N@oq%1OErBh&;RDt}pg!2Jd59)OC_GReL@G}9Bhf1$OtD~poO-{D<^~ZlL=-tDM1=`T0j#qprlM460NG8WE11<QFthFNEo4mF$h1f7fG+v{lnF|bE3qKj?Q$R-sA7S^K^vs7V3nRb!oTD*bq(JhQQ8pJ9t;+Om;vnA%NQ(ZV8ZL0K?l$%DH}*aBuh1{P<@~^E`KXQunq=}xdM`#l4<JQY0pF)$OwGtl!Ouj+CXnCbql!ML|_zwhMC7O|JWnt>btOhA^7>4IC&~L9L+1SE_lbk=p8mOHNs6Bq#}$I`m-_k{vd#2GjvRX)lH`kv&aFPF1Ywke(vX2v~+82w+bi5<Wts6IeJ+N4+j(WN%)U@6S&Jil9-t(Js%B{_o|uNvm|fK<d1HF3tGM;Gt;qjDTm(cw<3B0(J}1SfnZG<Xya}|93T}RIh!*XUpbvVo=nb$p7LbDn~utIR$c945r7Fp3>t*tZSbX|JF|TO9~YA)5KBzE7>NxQAl(jkjJCDXlJ}8hac#piRD{G`LOVS1cBkG07LVpB;u)J7p#cVR%TQGB;?yW&S@Ox<hxQRUE3{XnzWlPSCGqn|o9Ax4?Dw0yd-Jlnvk1TFDH0NdVc7)$!@)_`)k5e*i}3|>N?uZLe6p(;nKSik(202=QWO$t*y{b*l1jQ|-gg-OcipEfvGa!%M`Z?OSqJ|fqtg7c4POTsTKo*a>IKE}$45~aW8gOfcPF`spQ9aXd(52*Xj4_YP;3h`xjU%sm0`><IMEx-+M|ttL@F#MLK&1f$f^-)Hr%|?OranW{KgnfOp#E+|0r5j;;|X-g6ATFE!}0w0JcI*F97B+tpJP>T|QLX@xXiNt?pk-3pN4T-h|f@#mi(3tpW=0-Hg{i-EV`JO`|H`f-ye7s^T)BfiUa{(B=3;N{Wmzp8{wt3Fy$<qI#T72K%TZq0HouG2~`;d6~Q?M+9D$o9OWIA7-wpcL8r|HbHix%jc6PL+|s$FUW}p{`9_7pTG3}Fqt^@wgYIt0-SaV^6#P*Z=;qzP7BN-^nc#)J1lMIk}?*75Jg~Sg>{cuI8InNd<<^jb{v9%I+JEupUiy}-}FOn>IQ`*`@IVF7}PZw?a>6>hewk`{Ldr&&)?&Jo(za{EM$BmHG_-96zTz_0I@jZr>$0oa;hPm4f-r~_T#mwe(j@fwj6>ghetUmJR5X{58{VQ8)>Imc6PLJvZ;#U;}+1DiD+RP4Wq+`2tR^n49`hG4wQcC{TYHv6mFp)IqhA-Pe9rvV82{|y@h5D#q>qYMg(h;a(pN6nQ>2WNa7?r0+NboQL(m^?M#V58$mkKf>8t%5|>UxU)l2=GulO%Ztk~SG!-C;McK5OdS%)z9cmHDqTBB?;|()uCAjUHxh&M#9i+DA6AI=u8vD*2vde6tuF_^%`NRea68QLm|5T@d^uWlJuDJj~8pAL1#Rn6+Pt5{c3?WItKh{wSQ^ePUAq&CG=i$&eY=|0nTcy!Dl=>hLBQ%;e6h1Dq8F%UlUT`2UO~9IwljscnwEazc&AcpVGL}0_)2Td|prxq+BowXY=9DRH2UMngcs96FhB!hIb_rG3#W!Vv5(Cykl_cJBeSQs^b4TK)SW~QX<zKh`<D02>4KHqdd-XPG*i(~o<Bah(9NSr^Q%eD5bZKRHl;K>0aKs_jz}RM?iitZc&@nc@FGau(>5F$ds9(l7yP6`jt2zy(X{&7}3W5muz!PO&Lv$G<_<Q5>=%zzT7(am>e&rRlC<owdF9fL$-@ivA;;BlvZsNuH4t8JI9DrU|9><z>fL=_7<{&L{#K@Ry84%_4QZzB%tBj58JVRKxiJ>Cp_JB)&8$mh<FhEFzM0iq$E0~;>ITJ$p9cIc{7~jZ6GN3J{!{_M115+j?rr=<6TY>3^+qc4mHP~3n@HQ&WDVdKmeCYWSgP}b)UeXdX;nfud9>wS(z$Y`Jka$3ENKg>opPrV2Q2>)1bwAVPB;3xGgzz=R0gQgQQOFfDAJdW>$;Rk{qLPxY(;SLih^<8*Tdh#B3-+Kb8M4=ZH)}YKa@o>z960r6ABkXhfvsiv6oVSTrdfPRbP-B5FRsR>UT5i{IjPmAu*x71MAFy*Ga}9Bf&YLx=$4Gbg$!lyREiiAD?qpN1Z+#hC+3b$&vf=zkk25bF*^IPn%AY&cIps|C@3i8131V_Bo>pBX0HjZxkI*0RFykx^9K%Ta|iFG!dwIJ1F4I|8|3H#v5Yl`p`B>DJHxzu{_x$KZ@g_%LKWg-WAR6GR&2}DEDTEl^^MuAZ3DV2ELKu%gj?g7jsrQ{kfJ8@{5K^I`gv04YE`LR9`+3ZQd8d=oTGuDN-bbx#1IqY2&Im?Jv{)F;AM#Lg7lT33nYy|Y%IEvmZiu^4(kr|Brqy8_bCbwdNPi-jo~?=A-;{VBrj;DFDOf8trJKoTwm|2R<gq^XE42F<PZ~+!lCxh?uL#{RBj%Y$xm@7dp4XtlZjeBH)Zwg%Cz5tk3RLj6_rz9<3nmz26kjJ5o!|ir(O;oV{wU#Rfa<vgW4^<CFtvv3L=>Prk6++c2Fv%Qj>uwWSO@7StH0#O@^e$T~UD*xwuKXM0HASJKQ~cVCvF9Mw~iF#!7*XL?^Xi`bYumicUds@fzbgmUt3?z0zrU*inHQPXHqJw7WxKS(o$akaVfg#CXj{je1MByXh1J#Yj!%(9{a%zlMg8Q{Sf_>}V*DdnbL-z#Hlg38+hd1P!KOT_K`f4dtR?q3xt8DfP?8;&_k7*&E~*#BYYRCHUUZc3UpT^xw8g;ud@AJM;pSK9>IexNo0y+h1E=zQd1eTJ?Lau!bjR6#9uIi1gVQk`Pv4U_R#2d|fVz7*aeNtR!)172X<l`7WS}Kun+xY-`|m$cTJMEXUl$RA5`!LAI^e6%^<V^gMVVYq*5j9by=FW_nd!JX6Gc<iuU~)fZIn25E7N0I<5>WTJ;I{&BFQC5r69+%?nG$=Y`hPH5kz$FPttB$d}m|4$Jo!RQLRM<UaLZ`fSH{EkhA?l6n+M~7lKJ9xtGpQ`A`&U+&!5GldrE#^2L54mFHV5~Au<?NAFEa_wbS(FkSPN3)KkH-U9OF27ug0G;;H3YYDhO#<z*HOAFE=AFxIP+Di?(M)1tSVzB<5rMrJ;z9c2+B=We8!B+Sf^Jgx^^*;br}%lK|#F$f8~Jm8Ts`A<4WUaj|2U^Xm6$pOBH@WsY1H~2aDI33-a}xaK~lD(k6L`)4_?Fb{bm2d8$knJbwlT3r74ZT8vo1%$pcXc+jYK-LnEdWFnTpRcCSmVj%h=*J3UqXjHA3MYo`+)U@?`O!p_drv1hXUu<yxx-svxVwd?V<<9?#5sm)!=(<ZC2E%PS!K6htLJD1RR?S>X8@(hev<6Xk`K34M9r!u5a-a@}&Ej>`L?26ySt&&A1*`JcCd0~3U)nW_?diyd8VOByL~uB$nv<+w)*Ip1;!VJ%XsA56OjPL)`2%;Xz!qp609?6&BUA{x2A1bQAde3!3d<nbRnkP%<%t*o4bk8WLJEe4p=+b^xp`lI`9{O>e9doGovv#H>7Y(MRq5-SRnS<U{JYda`!-%-9IGT354vWnWZgBE(3oBEQ`dO%c-d4$8Ks^=t4<sPn5trWSzcP|`cmKppV$Omb=5`syDK=FM}gw2a?I=i{dVAr;du!DARl#*R~oSLRI9)mV?c(@WsJ~rO>|K279db1OOEZ8vu_!QM?gHL#I3`--4=S!?1pM;@)VQER8U0rPSMy~n@m<Tr!NkB`&g<Sxs+j2`6Nu|yT(!7G&)k5o!&62BLON?K$czDtHMnjE2{9X(<yggBCR1Ix1-~1`tWu@9-)hf48%-pxDPj380tLTBwLA-DrP~DnARymKXa}e%}>!S`~i{;)ZV1Tr`~Ja%NDfT@B$#_2DDwn;$l27Bf3wi5`(P{Mk44hpc81-nyTJ63D;|2wFaFfv;9zwADTw%;KU_Wz5Z5bhi*}@+%SLbN6Wl5vNkQLQ1-5a!PPg|Z!a5NL2G1HKncBNz1@{i|HA~HA~H0&LDzyj@k=3DzN!JDm;7|LchQsw5WO6Efgv5?ElRTiz5tC@!=^j9fjo6qLIr3Tm5U@1{sPvDrFFS6bMA&KgvPVrIzDZOVKiO;US}ct2O?dKwl+HpVmmbMd;m{u=(s$Bj4$an@m~tY;W1qE*bd)90csb&jN!HiD252MXbc5Yiu14ZQ!#3(qDE(CIEOE+0FV=#&&62S-MrW$;3+iqpa%5WjJsm=6m(3mpJ7yIG@|5hGSHyRpib#MQh*iG?J3CAJMG$$Tl<h5NNA;r+k45a;Ha(Z%&bJf3pGL+wBi8*<wvt83cfBFdJ@~X1HdEO=isHl5lVf_0h~Wc>74T_Dezo#wAh5q8eo=$TrFMF3FtKS_qa9LIh^Xuyg$Bq{r%hHH{r?g+mrAwub&@V9tBPYD`9XR?OOw+Z7|2^1e)Q6zzYZ_p|+r7QDUJtl48g+cd#qe_D_R|)2r?*jm*rsv|KxTli{&cNgi+J-6Gv-$PW@go%Q2R^gD#zC4TSIO#>5aZVH!38}ekW8DKnIHQ7U8vO=j%+gL*MI`>#rF@r5MpAvM45Y~9p0qfXc3z0I|uu7xA7Je33LzFDdI|g!5xCf?WO}ETIr-unOR3)*cP^MTpDNeBRSboXZ4bYIfCPw$OnFK2%kVWL&jFq=*DY+4VAVq|Fp$rhhsn4H)b&TJJ&BkY+w5$xF^z({;VeN72x)wKsiKFg#WLpd(xJ0~d9yCPuFk5Y7@MV_FfUm@HiLpcsBrjsHgGBnBkJpP*PmCjrKF=kRr~6A41XMIC4Or@ellDx_W<=_jOTq7wL5Mxo4MTwhrOO$p8g+0wIm2GMq2e|C)MjkK6_U5*3!A%f>rqWlM+axx>KI6GRxp=xWVI+rT@5-Pb}5zpkO%*{F3{^NSkOgTT+!W#!*rA$J8olXl$!pLb%b4Vs)lpt4aCJKbh@lW2@KFmf{=eaBEVB@09k`*>b@~YBBOa&#~e(yY1J5aPD<Rs63qd{(|H<Y=e~^THR?!|H8-_*&uo=N7kKCe4`p3O1>P;-$o(`Hqr1q6gmw@e@XosNLUBK(IC1eO#V`MnZM<cD_3O|7RC(u{U;Ybxwe;7&{ri>oKmYu{|M~0B|F-b5Uw{6W1lqut`7i&PK+*WGfBWxh+_-lK%IcQr)euhG3`)Trv@s#KQ-CT^+DP1!0q!5Y1+xUr&o~r;FggW>;ID~ZQ>BT=^bEcUY<0$4su}qQ`+1ze4Kl?EH1wvh0jLUU1A?dWvs}O~k2nq0duZi1F7h!8`UC1{v#xz14@Zd_cjmZ~t8|akt?uZdxnpZ_pNw~v*B)HO;B9KCQ&cT%bwe?Ce2eF{4<1k2rR;6uaZF1)je=MWpYZ`^B=p5uL(jbaqIJH9PRo_@f;&)NP>pMx>{Pef31F(o*wy(mo-_B9^$vTD>qPedKAu0PZ{NdYlveCLpzXdzLf-x$NA>$Imf*F@z_9bagGPLLO!0ZDlkP~I$~DH7kR}t^f%(wxL(0cgVa_H1aUDp`!)%7xtZlFuu)C(rmPwO{?vvMjO3k>sP8_#%PNk_lF2S+|ce|`0kv`~OjKwE`<ZqLM2XIYIm83yiqwQaC$4Z6H6xYN+O?)DctJ#NhTcz%Ne2f>($0>xtYi)0`1wELh3)1LmY2C$g7cRC-i0HCi_cb;>focU|RGkXziC&!X1O7;s_P!&QXIdI(bP<rlZtrD3z0>La_{=ct^_W~O$k2#3T<Zu`A>gBc6<4iBPyBsl6iU25xQybrOz_*J|E<oDV&REewDR^<O-e@zNp#KH>}I^n*2aQ)aoBZ$cW2?oyFG`%Lm(bX3V#em8Ge|bH{)4gBT3Z;*?nUo0gAi<e9$3A&2k#j7Vu=C=XO;t`r6N@9>wb*yD!+r$%r3P)Z(b#*mFurtx|>}`5@n{%lNmC2RSaCkr0J_Oy^_qS6{K}^^Ln2b?<>XL3+l3JU9!A?*~z9-5XamkIdZ{r{Fpr8RHb){dOdouOxu~8E5S{VkNO5=ZNGeTh+fN-Wn~Xd*>SDBaccc%l-e+IVt1vkN$&Aq6UbU(C|zfevzvRf7*A=n!H5Q83Lm)WUv3!=_+>lO6_Wmy@vp-Q~0g!k8gdH01oRLmjUukUbe`qeV+=0j9}NBn2g}a)91_7)zd3LIbM0EE{Mf4HcY{y&mr#|R@J6D*rjKIM~5jwC68p<F?dNMY|*BW?4kS*-RyQ}GO(XKkfHl_AIH#@P<rwbeSr}&9XewR-#vRBK7aT6$k+7_rP>t~Xo*`Yh0??D>mgN&N;i4$FovhW0Bqj(k6yfc79PEL_K&_DNN%2SQRu)xhPpgXlwkb%#p_pp3e~U!_wwv|J1NrL<#sxK$far`j1$gb^DM4Z`I>fFW>8UQo7=MkeB&tAJ8Txh8T|eXumHAW3URDF#mhyg$|=R$*N<oh^4hwmIOiRed=>Z-1B8%ACcjegKBob!vyaZ&I?~3fTCP#8q)OtTT&Xv`Awy=Z8oGAbl1g|OMGs{wH6*!0R#Vv3$g14AJ27M)J8LE01sk$XAp6a_#;kF<pXsnm-(30a9qkOqUhvPDF8y7oyW!u8N8Mc(UClJ?u^{|w@x`<Rx$zjT=nkP2o)mrHk4BPZ;I6Gp{}%1xI&M4~JzB;WqjIwy)fAt<MMvSDj<AG(c5wJ)GW=pPX`C6-g4D1_x?p05k&rw*;P1RH3d6ldT@gj!VJ?pj{aX%_<)ac*+&q8ZI^txAc?U4@F_iZNv1f1fbl^@$JfJm*>KeU0KZdu3-)@aDl0~$O<_Q_z{yNXsQ*bXKgtLYdP%Fx2Sgt0Ntv&nZ8{f<u<Z#nic99f$cIv-94v(H4y*LiPeRlFfg(Fa~*yBO@8cW87u_2`68;5ne!JfSQ%kk@XC*m&JgTsk$UJop|eI^+lqCD$RD6X0IjW|Zm><&%*{)=CBc+4g|zl0k_!Eq9%kuYVa#{XBt_~^~?vy<cS)$8#4m#?0`{{E)LH#5aIVl%O_E=nadB{AS-&G@hU|GOw>{P(vBd9^|sGA9*hbcaVHNz`3~rYu51G>R*81=_WZX4O<zc1ranc@giPHuDhQam!W}W`8)GjoKWncjzQ;SZxxW*`jX}+=34q`X)lLyuk|Q`qpK}Q){m_5!pl=(5~jhkgQCsGbmy@51{@G>ZaF?h`@eVNxHw+x>m<E8k$hz2b7BL^O(eKjr~`G?q`wKTG?ZB1v}iqR7eY0)hVsYlvpHn?^-R?b>VJbW!+saGu?pA041G9{(Ikzv-@tE)kPF{N<vshrNzN_xR=zjL!DWSMr7ZC3U&8X>&87c&H35{x>8S9&rqTXjel{E1eGV0?u+r1&F^rt4Z{=F^q%|L9Jih|ZYIY&+_~!Ayvlvhdx=kja^;Lsqp&CvrZY@Q@uz-(zfbt$_VWE9=LGLo%joLus7{j;qUuGvG+389Hj7)W;>KoB@z(W?UGWOD#IYkUey1wO1_O$dD4hLkU1;0Tp8p34W>0('


import base64
import hashlib
import importlib.util
import zlib

def sha256(body):
    return hashlib.sha256(body).hexdigest()

sample_bytes = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in SAMPLES).encode('utf-8')
crawler_bytes = zlib.decompress(base64.b85decode(CRAWLER_PAYLOAD))
if sha256(crawler_bytes) != CRAWLER_SHA256:
    raise ValueError('Snapshot crawler bị hỏng.')
if (len(SAMPLES) != EXPECTED_URL_COUNT or len({r['domain'] for r in SAMPLES}) != EXPECTED_DOMAIN_COUNT
        or len({r['url'] for r in SAMPLES}) != len(SAMPLES) or len({r['id'] for r in SAMPLES}) != len(SAMPLES)):
    raise ValueError('Danh sách mẫu thiếu hoặc trùng ID/URL.')
manifest_path = RUN_DIR / 'manifest.json'
sample_path = RUN_DIR / 'sample.jsonl'
crawler_path = RUN_DIR / 'crawler_snapshot.py'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (saved.get('probe_kind') != PROBE_KIND
            or saved.get('sample_sha256') != sha256(sample_bytes)
            or saved.get('source_sha256') != SOURCE_SHA256):
        raise ValueError('Run cũ không khớp mẫu của notebook này.')
    if not sample_path.exists() or sample_path.read_bytes() != sample_bytes:
        raise ValueError('Danh sách mẫu của run cũ đã thay đổi.')
    if not crawler_path.exists() or sha256(crawler_path.read_bytes()) != CRAWLER_SHA256:
        raise ValueError('Snapshot khác run cũ; hãy tạo run mới.')
    if json.loads((RUN_DIR / 'source_profile.json').read_text(encoding='utf-8')) != SOURCE_PROFILE:
        raise ValueError('Source profile của run cũ đã thay đổi.')
else:
    sample_path.write_bytes(sample_bytes)
    crawler_path.write_bytes(crawler_bytes)
    (RUN_DIR / 'source_profile.json').write_text(
        json.dumps(SOURCE_PROFILE, ensure_ascii=False, indent=2), encoding='utf-8')
    manifest_path.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'probe_kind': PROBE_KIND, 'dataset': 'AIGuruTinix/ViBioMIR',
        'source': 'embedded samples from data/links_corpus.parquet',
        'source_sha256': SOURCE_SHA256, 'source_rows': SOURCE_PROFILE['rows'],
        'sampling': SAMPLING_METHOD, 'seed': 42, 'selected': len(SAMPLES),
        'selected_domains': EXPECTED_DOMAIN_COUNT,
        'baseline_archive': BASELINE_ARCHIVE, 'baseline_archive_sha256': BASELINE_ARCHIVE_SHA256,
        'sample_sha256': sha256(sample_bytes),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

spec = importlib.util.spec_from_file_location('domain_probe_crawler', crawler_path)
crawler = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = crawler
spec.loader.exec_module(crawler)
priority_path = RUN_DIR / 'priority_probe_snapshot.py'
priority_bytes = zlib.decompress(base64.b85decode(PRIORITY_PAYLOAD))
if sha256(priority_bytes) != PRIORITY_SHA256:
    raise ValueError('Priority probe snapshot bị hỏng.')
if priority_path.exists() and sha256(priority_path.read_bytes()) != PRIORITY_SHA256:
    raise ValueError('Priority probe khác run cũ; tạo run mới.')
priority_path.write_bytes(priority_bytes)
priority_spec = importlib.util.spec_from_file_location('domain_priority_probe', priority_path)
priority_probe = importlib.util.module_from_spec(priority_spec)
sys.modules[priority_spec.name] = priority_probe
priority_spec.loader.exec_module(priority_probe)
if not isinstance(USER_AGENT, str) or not USER_AGENT.strip() or any(c in USER_AGENT for c in ('\r', '\n')):
    raise ValueError('USER_AGENT phải là chuỗi header hợp lệ.')
crawler.UA = USER_AGENT
crawler.ROBOT_AGENT = USER_AGENT.split('/', 1)[0]
(RUN_DIR / 'baseline_domains.json').write_text(json.dumps(BASELINE_DOMAINS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã nạp {len(SAMPLES)} URL mẫu và crawler {crawler.VERSION}.')


## 4. Chuẩn bị bảng kết quả và phân loại lỗi

Output riêng: `vao_duoc.csv`, `khong_vao_duoc.csv`, `chua_chay.csv`,
`loi_theo_loai.csv`, `can_xem_noi_dung.csv`; bảng tổng là `ket_qua.csv`.
Các CSV dùng UTF-8 BOM để mở được tiếng Việt trong Excel.
`error_detail` giữ nguyên lỗi kỹ thuật; `reason` diễn giải tiếng Việt.
Mở `ket_qua.html` để lọc theo domain hoặc trạng thái.
`uu_tien_fix_domain.csv/html` xếp hostname còn lỗi theo số URL dataset giảm dần.
`tong_hop_domain.csv` gồm cả hostname đã phục hồi; `WEB_ERRORS.md` là báo cáo lỗi chi tiết.


In [ ]:
import csv
import html
import shutil
from collections import Counter
from IPython.display import HTML, FileLink, display

STATUS_REASONS = {
    'robots_unknown': 'Không xác định được chính sách robots.txt; crawler dừng trước khi tải trang.',
    'robots_denied': 'robots.txt trả HTTP 401/403; crawler dừng theo chính sách robots.',
    'robots_disallowed': 'robots.txt không cho phép crawler tải URL này.',
    'http_error': 'Trang đích trả HTTP ngoài khoảng 200–299.',
    'network_error': 'Không tải được trang do lỗi kết nối sau retry.',
    'unsafe_address': 'Địa chỉ đích không phải IP public được crawler cho phép.',
    'invalid_url': 'URL không hợp lệ hoặc dùng cổng ngoài 80/443.',
    'oversized': 'Phản hồi vượt giới hạn kích thước của crawler.',
    'response_deadline': 'Tải phản hồi vượt giới hạn thời gian tổng.',
    'rate_limited_deferred': 'Server yêu cầu chờ Retry-After quá lâu; crawler hoãn URL.',
    'redirect_limit': 'Vượt số lần redirect cho phép.',
    'unsupported_content_type': 'Đã tải phản hồi nhưng định dạng không được hỗ trợ.',
    'unexpected_error': 'Lỗi ngoài dự kiến; xem error_detail.',
}

def classify_error(status, detail, http_status):
    lower = detail.lower()
    if status == 'http_error':
        return f'page_http_{http_status}'
    if status == 'robots_disallowed':
        return 'robots_disallowed'
    if status.startswith('robots_') and 'robots http ' in lower:
        return 'robots_http_' + lower.split('robots http ', 1)[1].split()[0]
    if 'html instead of robots.txt' in lower:
        return 'robots_html_instead_of_txt'
    prefix = 'robots_' if status.startswith('robots_') else ''
    if 'waiting for domain lock' in lower:
        return prefix + 'domain_lock_budget'
    if 'request budget exhausted' in lower or 'retry budget exhausted' in lower:
        return prefix + 'budget_exhausted'
    if 'winerror 10013' in lower or 'network is unreachable' in lower:
        return prefix + 'environment_network_blocked'
    if any(s in lower for s in ('timed out', 'timeout')):
        return prefix + 'timeout'
    if any(s in lower for s in ('sslerror', 'certificate verify failed', 'ssl:')):
        return prefix + 'ssl_error'
    if any(s in lower for s in ('nameresolutionerror', 'getaddrinfo failed', 'name or service not known', 'failed to resolve')):
        return prefix + 'dns_error'
    if any(s in lower for s in ('connection refused', 'failed to establish a new connection', 'connection reset')):
        return prefix + 'connection_error'
    return status

def result_rows():
    rows = []
    for sample in sorted(SAMPLES, key=lambda s: (-SOURCE_PROFILE['host_counts'][s['domain']], s['domain'], s['sample_index'])):
        record_path = RUN_DIR / 'records' / f"{sample['id']}.json"
        record = json.loads(record_path.read_text(encoding='utf-8')) if record_path.exists() else {}
        if record and (record.get('id'), record.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Checkpoint không khớp ID/URL nguồn.')
        status = record.get('status', 'pending')
        page_http = record.get('http_status')
        robots = record.get('robots', {})
        # Robots failure may happen before fetch() attaches its evidence to the record.
        if status.startswith('robots_') and not robots:
            from urllib.parse import urlsplit
            target = record.get('requests', [{}])[-1].get('url', sample['url']) if record.get('requests') else sample['url']
            if record.get('redirects'):
                target = record['redirects'][-1]['to']
            p = urlsplit(target)
            robot_file = RUN_DIR / 'robots' / (sha256(f'{p.scheme}://{p.netloc}'.encode()) + '.json')
            if robot_file.exists():
                robots = json.loads(robot_file.read_text(encoding='utf-8'))
        detail = record.get('error', '')
        browser_file = RUN_DIR / 'browser_records' / f"{sample['id']}.json"
        browser = json.loads(browser_file.read_text(encoding='utf-8')) if browser_file.exists() else {}
        if browser and (browser.get('id'), browser.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Browser checkpoint không khớp ID/URL.')
        requests_access = ('CHUA_CHAY' if status == 'pending' else
                  'VAO_DUOC' if page_http is not None and 200 <= page_http < 300 else 'KHONG_VAO_DUOC')
        if DIRECT_PUBLIC_PROBE:
            access, effective_engine, effective = priority_probe.access_outcome(record, browser)
            if record.get('challenge_detected'):
                requests_access = 'KHONG_VAO_DUOC'
        else:
            access, effective_engine, effective = requests_access, 'requests', record
        flags = record.get('quality_flags', [])
        if status == 'pending':
            reason, error_type, stage = 'Chưa chạy URL này.', '', 'pending'
        elif status == 'extracted':
            stage = 'done'
            error_type = ('qc_' + record['quality_status'] if record.get('quality_status') in ('review', 'rejected') else '')
            reason = ('Tải được và nội dung đạt QC tự động.' if record.get('quality_status') == 'accepted'
                      else 'Tải được, nhưng nội dung cần xem lại hoặc bị QC loại: ' + ', '.join(flags))
        else:
            stage = ('robots' if status.startswith('robots_') else 'extraction' if access == 'VAO_DUOC' else 'fetch')
            error_type = record.get('error_kind') or classify_error(status, detail, page_http)
            reason = STATUS_REASONS.get(status, status)
        requests_reason = reason
        if effective_engine == 'selenium':
            stage = 'done_browser'
            reason = 'Requests lỗi; Selenium tải được trang. QC: ' + effective.get('quality_status', 'unknown')
        rows.append({
            'domain': sample['domain'], 'dataset_url_count': SOURCE_PROFILE['host_counts'][sample['domain']],
            'sample_id': sample['id'], 'sample_url': sample['url'], 'access_result': access,
            'requests_access_result': requests_access, 'effective_engine': effective_engine,
            'effective_http_status': effective.get('http_status'),
            'effective_quality_status': effective.get('quality_status', ''),
            'effective_char_count': effective.get('char_count'),
            'effective_quality_flags': '; '.join(effective.get('quality_flags', [])),
            'http_status': page_http, 'robots_http_status': robots.get('status'),
            'robots_attempt_count': robots.get('attempt_count'),
            'robots_cache_hit': robots.get('cache_hit'),
            'robots_fetch_id': robots.get('fetch_id', ''),
            'robots_error_kind': robots.get('error_kind', ''),
            'robots_elapsed_seconds': robots.get('elapsed_seconds'),
            'robots_lock_wait_seconds': robots.get('lock_wait_seconds'),
            'crawl_status': status, 'failure_stage': stage, 'error_type': error_type,
            'reason': reason, 'requests_reason': requests_reason,
            'error_detail': detail, 'quality_status': record.get('quality_status', ''),
            'quality_flags': '; '.join(flags), 'char_count': record.get('char_count'),
            'title': record.get('title', ''), 'final_url': record.get('final_url', ''),
            'elapsed_seconds': record.get('elapsed_seconds'),
            'reused_from': record.get('reused_from', ''),
            'request_attempt_count': len(record.get('requests', [])),
            'retryable': record.get('retryable'), 'retry_deferred': record.get('retry_deferred', ''),
            'page_attempted': bool(record.get('requests')),
            'browser_status': browser.get('status', ''),
            'browser_http_status': browser.get('http_status'),
            'browser_quality_status': browser.get('quality_status', ''),
            'browser_char_count': browser.get('char_count'),
            'browser_final_url': browser.get('final_url', ''),
            'browser_error': browser.get('error', ''),
            'browser_attempt_count': browser.get('attempt_count'),
            'sample_role': sample['sample_role'], 'sample_index': sample['sample_index'],
        })
    return rows

def domain_rows(rows):
    result = []
    for domain in sorted(BASELINE_DOMAINS, key=lambda h: (-SOURCE_PROFILE['host_counts'][h], h)):
        selected = [r for r in rows if r['domain'] == domain]
        failed = [r for r in selected if r['access_result'] == 'KHONG_VAO_DUOC']
        content = [r for r in selected if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted']
        pending = sum(r['access_result'] == 'CHUA_CHAY' for r in selected)
        errors = Counter(r['error_type'] for r in failed)
        errors.update('qc_' + (r['effective_quality_status'] or 'unknown') for r in content)
        result.append({
            'domain': domain, 'dataset_url_count': SOURCE_PROFILE['host_counts'][domain],
            'baseline_error_type': BASELINE_DOMAINS[domain]['error_type'],
            'selected_urls': len(selected), 'additional_urls': sum(r['sample_role'] == 'additional' for r in selected),
            'missing_additional_urls': BASELINE_DOMAINS[domain]['shortfall'],
            'completed_urls': len(selected) - pending, 'pending_urls': pending,
            'page_not_attempted': sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in selected),
            'accessible_urls': sum(r['access_result'] == 'VAO_DUOC' for r in selected),
            'access_failures': len(failed), 'content_issues': len(content),
            'total_problem_urls': len(failed) + len(content),
            'additional_problem_urls': sum(r['sample_role'] == 'additional' for r in failed + content),
            'error_types': '; '.join(f'{kind}: {n}' for kind, n in sorted(errors.items())),
            'test_status': 'CHUA_CHAY' if pending == len(selected) else 'CHUA_HOAN_TAT' if pending else 'CON_LOI' if failed or content else 'DA_PHUC_HOI',
        })
    return result

def write_csv(name, rows, fields):
    temporary = RUN_DIR / (name + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(RUN_DIR / name)

def save_outputs(show=True):
    rows = result_rows()
    fields = list(rows[0])
    counts = Counter(r['access_result'] for r in rows)
    summary = {'selected': len(rows), 'vao_duoc': counts['VAO_DUOC'],
               'khong_vao_duoc': counts['KHONG_VAO_DUOC'], 'chua_chay': counts['CHUA_CHAY'],
               'crawl_status_counts': dict(Counter(r['crawl_status'] for r in rows)),
               'quality_status_counts': dict(Counter(r['quality_status'] for r in rows if r['quality_status'])),
               'effective_quality_status_counts': dict(Counter(r['effective_quality_status'] for r in rows if r['effective_quality_status'])),
               'created_at': datetime.now(timezone.utc).isoformat()}
    domains = domain_rows(rows)
    priority = [dict(priority_rank=i, **r) for i, r in enumerate((r for r in domains if r['total_problem_urls']), 1)]
    summary.update(hostnames=len(domains), domains_with_errors=len(priority),
                   priority_order='dataset_url_count descending', user_agent=USER_AGENT,
                   page_not_attempted=sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in rows),
                   additional_urls=sum(r['sample_role'] == 'additional' for r in rows))
    summary.update(request_mode='direct_public_page_probe' if DIRECT_PUBLIC_PROBE else 'robots_enforced',
                   requests_http_2xx=sum(r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' for r in rows),
                   browser_tested=sum(bool(r['browser_status']) for r in rows),
                   browser_http_2xx=sum(r['browser_http_status'] is not None and 200 <= r['browser_http_status'] < 300 for r in rows))
    write_csv('tong_hop_domain.csv', domains, list(domains[0]))
    write_csv('uu_tien_fix_domain.csv', priority, ['priority_rank', *domains[0]])
    (RUN_DIR / 'uu_tien_fix_domain.json').write_text(json.dumps(priority, ensure_ascii=False, indent=2), encoding='utf-8')
    write_csv('ket_qua.csv', rows, fields)
    write_csv('requests_vs_selenium.csv', rows, fields)
    for value, filename in [('VAO_DUOC', 'vao_duoc.csv'), ('KHONG_VAO_DUOC', 'khong_vao_duoc.csv'), ('CHUA_CHAY', 'chua_chay.csv')]:
        write_csv(filename, [r for r in rows if r['access_result'] == value], fields)
    write_csv('can_xem_noi_dung.csv', [r for r in rows if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted'], fields)
    failures = Counter(r['error_type'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC')
    errors = [{'error_type': kind, 'domain_count': len({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}),
               'url_count': n,
               'domains': '; '.join(sorted({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}))}
              for kind, n in failures.most_common()]
    write_csv('loi_theo_loai.csv', errors, ['error_type', 'domain_count', 'url_count', 'domains'])
    (RUN_DIR / 'ket_qua.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    md = ['# Các website còn lỗi sau lượt test bổ sung', '',
          f"Thời điểm UTC: {summary['created_at']}. Run: `{RUN_DIR.name}`.", '',
          f"Đã chạy {len(rows)-counts['CHUA_CHAY']}/{len(rows)} URL thuộc {len(domains)} hostname. "
          f"Tải được sau Requests/Selenium: {counts['VAO_DUOC']}; chưa tải được: {counts['KHONG_VAO_DUOC']}; chưa chạy: {counts['CHUA_CHAY']}.", '',
          f"Requests tải được: {summary['requests_http_2xx']}; Selenium phục hồi thêm: {summary['browser_recovered']}. "
          'Các cột lỗi Requests vẫn giữ nguyên dù Selenium phục hồi thành công.', '',
          f"{summary['page_not_attempted']} URL đã xử lý nhưng chưa request bài viết vì dừng trước bước tải bài. "
          'Các trường hợp này được phân biệt với bài viết thực sự trả HTTP lỗi.', '',
          f'HTTP User-Agent: `{USER_AGENT}`.', '',
          f"Bộ mẫu có {len(rows)} URL thuộc {len(domains)} hostname, gồm {summary['additional_urls']} URL bổ sung. "
          'Gồm lỗi truy cập và lỗi nội dung/QC. Số URL dataset là số dòng của hostname, chưa loại URL trùng.', '',
          'Ưu tiên theo số URL dataset giảm dần; không suy ra toàn bộ URL của domain đều bị lỗi. '
          'Dừng ở robots là chưa request bài viết, không đồng nghĩa website không mở được trong trình duyệt.', '',
          '| Ưu tiên | Hostname | Số URL dataset | Mẫu đã chạy | Lỗi truy cập | Lỗi nội dung | Loại lỗi |',
          '|---:|---|---:|---:|---:|---:|---|']
    for item in priority:
        md.append(f"| {item['priority_rank']} | {item['domain']} | {item['dataset_url_count']:,} | {item['completed_urls']}/{item['selected_urls']} | {item['access_failures']} | {item['content_issues']} | {item['error_types']} |")
    md += ['', '## Domain đã phục hồi trong các mẫu', '']
    md += [f"- {r['domain']}: {r['accessible_urls']}/{r['selected_urls']} URL tải được và đạt QC." for r in domains if r['test_status'] == 'DA_PHUC_HOI'] or ['Chưa xác định domain phục hồi hoàn toàn.']
    md += ['', '## Domain thiếu URL bổ sung', '']
    md += [f"- {r['domain']}: lấy được {r['additional_urls']}/4 URL mới; dataset có {r['dataset_url_count']} dòng." for r in domains if r['missing_additional_urls']]
    md += ['', '## Chi tiết lỗi theo domain', '']
    for item in priority:
        md += [f"### {item['priority_rank']}. {item['domain']}", '',
               f"{item['dataset_url_count']:,} URL nguồn; {item['page_not_attempted']} mẫu đã xử lý nhưng chưa request bài viết. "
               f"Lỗi baseline: `{item['baseline_error_type']}`.", '']
        for row in rows:
            if row['domain'] != item['domain'] or row['access_result'] == 'CHUA_CHAY' or (row['access_result'] == 'VAO_DUOC' and row['effective_quality_status'] == 'accepted'):
                continue
            detail = (row['error_detail'] or row['quality_flags']).replace('\n', ' ').replace('`', "'")
            md += [f"- ID {row['sample_id']} ({row['sample_role']}): [URL](<{row['sample_url']}>) — `{row['error_type']}`. "
                   f"HTTP trang: {row['http_status']}; HTTP robots: {row['robots_http_status']}. {row['reason']} Lỗi gốc: `{detail}`."]
        md.append('')
    (RUN_DIR / 'WEB_ERRORS.md').write_text('\n'.join(md), encoding='utf-8')

    visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'sample_role', 'access_result',
                      'http_status', 'robots_http_status', 'robots_attempt_count', 'robots_cache_hit', 'error_type', 'reason', 'error_detail',
                      'quality_status', 'quality_flags', 'char_count']
    labels = ['Domain', 'Số URL nguồn', 'ID mẫu', 'URL mẫu', 'Mẫu cũ/mới', 'Vào được?', 'HTTP trang',
              'HTTP robots', 'Lần thử robots', 'Cache hit robots', 'Loại lỗi', 'Lý do', 'Lỗi gốc', 'QC', 'Cờ QC', 'Số ký tự']
    if DIRECT_PUBLIC_PROBE:
        visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'http_status',
                          'error_type', 'error_detail', 'quality_status', 'char_count',
                          'browser_status', 'browser_http_status', 'browser_quality_status', 'browser_char_count', 'browser_error',
                          'access_result', 'effective_engine']
        labels = ['Domain', 'Số URL nguồn', 'ID', 'URL', 'HTTP Requests', 'Lỗi Requests', 'Chi tiết',
                  'QC Requests', 'Ký tự Requests', 'Selenium', 'HTTP Selenium', 'QC Selenium', 'Ký tự Selenium', 'Lỗi Selenium',
                  'Kết quả cuối', 'Engine tải được']
    def esc(value):
        return html.escape(str(value if value is not None else ''), quote=True)
    body = []
    for row in rows:
        cells = []
        for field in visible_fields:
            value = row[field]
            text = (f'<a href="{esc(value)}" target="_blank" rel="noopener noreferrer">{esc(value)}</a>'
                    if field == 'sample_url' else esc(value))
            cells.append('<td>' + text + '</td>')
        body.append(f'<tr data-access="{row["access_result"]}">' + ''.join(cells) + '</tr>')
    style = """<style>body{font:14px/1.5 system-ui,sans-serif;margin:20px;color:#172b4d}
    .table-wrap{overflow:auto;max-height:70vh;border:1px solid #ccd4df}table{border-collapse:collapse;width:100%}
    th,td{padding:9px;border-bottom:1px solid #dce2e9;text-align:left;vertical-align:top}th{background:#e8eef6;position:sticky;top:0}
    td{min-width:95px;max-width:330px;overflow-wrap:anywhere}td:nth-child(4),td:nth-child(9),td:nth-child(10){min-width:260px}
    input,select{padding:8px;margin:0 10px 14px 0}a{color:#1654a0}</style>"""
    table = '<div class="table-wrap"><table><thead><tr>' + ''.join('<th>' + esc(label) + '</th>' for label in labels)
    table += '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>'
    heading = f'<h1>Kết quả thử crawl {len(rows)} URL / {len(domains)} hostname</h1><p>Vào được: {counts["VAO_DUOC"]} · Không vào được: {counts["KHONG_VAO_DUOC"]} · Chưa chạy: {counts["CHUA_CHAY"]}</p>'
    heading += '<p>Vào được tính cả Requests và Selenium phục hồi; trang challenge được loại. Các cột Requests vẫn giữ lỗi gốc, QC là kết quả riêng. Kết quả chỉ áp dụng cho các URL mẫu.</p>'
    controls = '<input id="search" placeholder="Tìm domain, URL hoặc lỗi" aria-label="Tìm kiếm"><select id="access" aria-label="Kết quả"><option value="">Tất cả</option><option>VAO_DUOC</option><option>KHONG_VAO_DUOC</option><option>CHUA_CHAY</option></select><span id="count"></span>'
    priority_fields = ['priority_rank', 'domain', 'dataset_url_count', 'selected_urls', 'completed_urls', 'pending_urls', 'access_failures', 'content_issues', 'error_types']
    priority_labels = ['Ưu tiên', 'Domain', 'Số URL dataset', 'Mẫu', 'Đã chạy', 'Chưa chạy', 'Lỗi truy cập', 'Lỗi nội dung', 'Loại lỗi']
    priority_heading = '<h2>Domain còn lỗi — số URL dataset giảm dần</h2><p>Gồm lỗi truy cập và lỗi nội dung/QC. Số URL nguồn không phải số URL đã xác minh lỗi.</p>'
    priority_table = '<div class="table-wrap"><table id="priority"><thead><tr>' + ''.join('<th>'+esc(label)+'</th>' for label in priority_labels)
    priority_table += '</tr></thead><tbody>' + ''.join('<tr>' + ''.join('<td>'+esc(r[f])+'</td>' for f in priority_fields) + '</tr>' for r in priority) + '</tbody></table></div>'
    (RUN_DIR / 'uu_tien_fix_domain.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Ưu tiên domain lỗi</title>'+style+priority_heading+priority_table+'</html>', encoding='utf-8')
    javascript = """<script>const rs=[...document.querySelectorAll('tbody tr')],s=document.getElementById('search'),a=document.getElementById('access');
    function filter(){let n=0;for(const r of rs){const ok=r.textContent.toLowerCase().includes(s.value.toLowerCase())&&(!a.value||r.dataset.access===a.value);r.hidden=!ok;n+=ok?1:0}document.getElementById('count').textContent=n+' hostname'}
    s.addEventListener('input',filter);a.addEventListener('change',filter);filter();</script>"""
    javascript = javascript.replace("document.querySelectorAll('tbody tr')", "document.querySelectorAll('tr[data-access]')").replace("n+' hostname'", "n+' URL'")
    (RUN_DIR / 'ket_qua.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Kết quả crawl domain</title>' + style + heading + priority_heading + priority_table + '<h2>Từng URL mẫu</h2>' + controls + table + javascript + '</html>', encoding='utf-8')
    # Archive outside the run directory to avoid including the archive itself.
    zip_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    if show:
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        display(HTML(style + priority_heading + priority_table))
        display(HTML(style + heading + table))
        print('File kết quả:', RUN_DIR)
        print('ZIP:', zip_path)
    return rows, summary, zip_path

# Create reports before networking, so partial runs have explicit CHUA_CHAY rows.
rows, summary, zip_path = save_outputs(show=False)
print('Đã chuẩn bị báo cáo và phân loại lỗi.')


## 5. Tải trang và đối chiếu Selenium

Chạy Requests trước, từng URL tuần tự trong một domain; các domain chạy song song.
Mỗi URL có timeout connect/read và ngân sách riêng, không tiêu hao ngân sách vì các mẫu cùng domain chờ nhau.
Ngân sách được kiểm tra giữa các bước; DNS và thử nhiều IP có thể làm thời gian thực vượt mức đặt.
Requests lỗi mới được thử Selenium theo `BROWSER_TEST_MODE`; có thể đặt `off` để chỉ chạy Requests.
Selenium Manager tìm/tải Chrome và driver. Nếu khởi động thất bại, lý do nằm trong `browser_setup_error.json`.
Requests và Selenium có kết quả riêng; DOM trống/challenge không được ghi thành tải được bài.
Requests retry tối đa 3 lần; lỗi mạng được đóng pool kết nối cũ trước lần thử tiếp theo.
Selenium dùng User-Agent thực và retry timeout tối đa 2 lần; 403/challenge không được retry liên tục.
`access_result` tính cả trường hợp Selenium phục hồi, còn các cột lỗi Requests vẫn giữ nguyên.
Để thử lại riêng nhóm lỗi, đặt `RECOVERY_RUN_DIR` là đường dẫn run cũ và giữ `RESUME_RUN_DIR = None`.
Recovery tạo run mới, kiểm tra hash raw 2xx trước khi reuse, để các checkpoint lỗi được tải lại.
Để chạy lại toàn bộ bằng cấu hình mới, giữ `RESUME_RUN_DIR = None` và Run All.


In [ ]:
import argparse
import os
import socket
import time

# DNS check is diagnostic only; failure here does not stop the crawl.
network_path = RUN_DIR / 'network_check.json'
if not network_path.exists():
    network = {
        'checked_at': datetime.now(timezone.utc).isoformat(),
        'environment': 'kaggle' if Path('/kaggle/working').is_dir() else 'colab' if Path('/content').is_dir() else 'local',
        'python': sys.version, 'platform': sys.platform,
        'proxy_environment_present': {key: bool(os.environ.get(key)) for key in
                                      ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY')},
        'dns': [],
    }
    for domain in NETWORK_CHECK_DOMAINS:
        start = time.monotonic()
        entry = {'domain': domain}
        try:
            entry['addresses'] = sorted({r[4][0] for r in socket.getaddrinfo(domain, 443)})
        except OSError as exc:
            entry.update(exception_class=type(exc).__name__, error=str(exc))
        entry['elapsed_seconds'] = round(time.monotonic() - start, 3)
        network['dns'].append(entry)
    network_path.write_text(json.dumps(network, ensure_ascii=False, indent=2), encoding='utf-8')
print('Network check:', network_path)

try:
    if DIRECT_PUBLIC_PROBE:
        if RECOVERY_RUN_DIR:
            recovery = priority_probe.seed_successful_requests(crawler, RUN_DIR, SAMPLES, RECOVERY_RUN_DIR)
            print('Recovery:', len(recovery['reused_ids']), 'URL reuse raw;', len(recovery['retry_ids']), 'URL thử lại.')
        records = priority_probe.crawl_requests(
            crawler, RUN_DIR, SAMPLES, workers=WORKERS, delay=DELAY,
            attempts=PAGE_ATTEMPTS, user_agent=USER_AGENT,
            timeout=(PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT), budget=PAGE_REQUEST_BUDGET, max_new=MAX_NEW)
        browser_samples = priority_probe.select_browser_samples(SAMPLES, records, BROWSER_TEST_MODE)
        if browser_samples:
            priority_probe.crawl_browser(crawler, RUN_DIR, browser_samples,
                                        user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY, delay=DELAY,
                                        load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS)
        crawler.report(argparse.Namespace(out=str(RUN_DIR)))
    else:
        crawler.locked_crawl(argparse.Namespace(
            out=str(RUN_DIR), workers=WORKERS, delay=DELAY, attempts=ATTEMPTS, max_new=MAX_NEW,
            robots_attempts=ROBOTS_ATTEMPTS, robots_connect_timeout=ROBOTS_CONNECT_TIMEOUT,
            robots_read_timeout=ROBOTS_READ_TIMEOUT, robots_retry_budget=ROBOTS_RETRY_BUDGET,
            robots_failure_ttl=ROBOTS_FAILURE_TTL,
        ))
finally:
    rows, summary, zip_path = save_outputs()


## 6. Xem lỗi riêng và tải output

`khong_vao_duoc.csv` chứa domain, URL, loại lỗi, lý do tiếng Việt và lỗi gốc.
`loi_theo_loai.csv` gom số hostname và số URL theo từng lỗi (HTTP 403, timeout,
robots trả HTML, robots cấm crawl, DNS, SSL…). File JSON trong `records/`
giữ trace request/redirect; `robots/` giữ bằng chứng robots.txt.
`network_check.json` ghi DNS của mọi hostname trong bộ mẫu để đối chiếu Kaggle/local;
`priority_probe_config.json` ghi User-Agent/header, timeout và chế độ tải trang.
`requests_vs_selenium.csv` và `browser_records/` giữ kết quả trình duyệt riêng;
`browser_setup_error.json` được ghi nếu Chrome/driver không khởi động được.
ZIP gồm kết quả, raw, ảnh chụp và snapshot để kiểm tra riêng.

Kaggle: tải ZIP trong **Output**. Colab: ô dưới hiện nút tải ZIP.
Jupyter local: dùng các liên kết file hoặc mở đường dẫn output được in ra.


In [ ]:
# Có thể chạy lại ô này để xuất báo cáo từ checkpoint mà không crawl thêm.
rows, summary, zip_path = save_outputs(show=False)
failed_rows = [r for r in rows if r['access_result'] == 'KHONG_VAO_DUOC']
print(f'Không vào được: {len(failed_rows)}/{len(rows)} URL, thuộc {len({r["domain"] for r in failed_rows})} hostname')
for row in failed_rows:
    print(f"{row['domain']} | {row['error_type']} | {row['error_detail']}")
print('\nOutput:', RUN_DIR)
for name in ['WEB_ERRORS.md', 'uu_tien_fix_domain.html', 'uu_tien_fix_domain.csv', 'tong_hop_domain.csv',
             'requests_vs_selenium.csv',
             'ket_qua.html', 'ket_qua.csv', 'vao_duoc.csv', 'khong_vao_duoc.csv',
             'loi_theo_loai.csv', 'can_xem_noi_dung.csv', 'chua_chay.csv']:
    display(FileLink(str(RUN_DIR / name)))
display(FileLink(str(zip_path)))

if 'google.colab' in sys.modules:
    import ipywidgets as widgets
    from google.colab import files
    button = widgets.Button(description='Tải ZIP kết quả')
    button.on_click(lambda _: files.download(str(zip_path)))
    display(button)
